# Bruise segmentation — FINAL results: analysis & visualisation

Everything below is derived from **one best-seed inference pass** on the 185-image
test set. For each model we take the seed that scored highest on **validation**
(chosen in `bruise_colab_final.ipynb`, saved to
`results_final/best_seed_val_selected/`), load that seed's checkpoint from
`runs_final/`, and predict once. SegFormer is scored at its val-fitted logit cut;
YOLO is scored **two ways** (native argmax + custom /255). Nothing here is a fresh
sweep — the operating points were fixed on val already.

Every chart prints the table under it, and each model keeps the **same colour** in
every figure (the palette is colourblind-safe). Read the annotation-ceiling section
(F) first — it reframes every other number.

| # | Section | Question |
|---|---------|----------|
| A | Accuracy & distribution | How good, and how spread out? |
| B | Safety / failure modes | *How* do they fail? Which path for YOLO? |
| C | Inference statistics | Which differences are real at n=28? |
| D | ⭐ Fairness (ITA skin tone) | Is it equitable across skin tone? |
| E | ⭐ Bruise size | *When* do they fail — and is size a fairness confound? |
| F | Annotation ceiling + speed | Do they beat human–human agreement? What do they cost? |
| G | Qualitative (optional GPU) | What does it actually look like? |

**This is best-seed (val-selected) analysis.** The 3-seed-averaged tables live in
`results_final/*.csv`; this notebook recomputes the best-seed slice fresh so the
figures, fairness and gallery all share one consistent inference pass.

## §1 · Configuration

In [ ]:
CFG = dict(
    img_size   = 640,
    zip_name   = "bruise_colab_final.zip",
    drive_dir  = "/content/drive/MyDrive/bruise_segmentation_gpu",
    work_dir   = "/content/bruise_final",          # local SSD, wiped on disconnect
    seeds      = (0, 1, 2),                          # only used to re-derive best seed if the CSV is absent
    workers    = 4,
    amp        = True,
    # sweep grids (only needed for the optional custom-255 re-derivation / fallback)
    cut_min = -6.0, cut_max = 6.0, cut_steps = 481,
    prob_min = 0.01, prob_max = 0.99, prob_steps = 197,
    render_gallery = True,   # set False to skip the optional GPU image gallery (section G)
)

SEGFORMER_MODELS = {
    "segformer_b2_teacher":   dict(arch="segformer", size="b2"),
    "segformer_b0_direct":    dict(arch="segformer", size="b0"),
    "segformer_b0_distilled": dict(arch="segformer", size="b0"),
}
YOLO_MODELS = ["yolo_sem_direct", "yolo_sem_distilled"]

# Display names + the fixed, colourblind-safe palette (same slot per model everywhere).
DISP = {
    "segformer_b2_teacher":   "SegFormer-B2 (teacher)",
    "segformer_b0_direct":    "SegFormer-B0 (direct)",
    "segformer_b0_distilled": "SegFormer-B0 (distilled)",
    "yolo_sem_direct":        "YOLO26n (direct)",
    "yolo_sem_distilled":     "YOLO26n (distilled)",
}
PALETTE = {
    "segformer_b2_teacher":   "#2a78d6",   # blue
    "segformer_b0_direct":    "#1baf7a",   # aqua
    "segformer_b0_distilled": "#eda100",   # yellow
    "yolo_sem_direct":        "#008300",   # green
    "yolo_sem_distilled":     "#4a3aa7",   # violet
}
# The 5 "core" variants used for model-to-model figures. YOLO uses its native-argmax
# best seed here (that's what the val selection picked); the custom-/255 path is added
# alongside only in the sections that explicitly compare the two YOLO paths.
CORE = list(SEGFORMER_MODELS) + YOLO_MODELS
print(f"{len(SEGFORMER_MODELS)} SegFormer + {len(YOLO_MODELS)} YOLO = {len(CORE)} core variants")

## §2 · Drive, GPU, dependencies

In [ ]:
import os, sys, time, json
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU (A100). Refusing to run on CPU.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q "transformers>=4.40,<6" "ultralytics>=8.4,<9" "albumentations>=2.0,<3" "scipy>=1.11" "pandas>=2.0" "matplotlib>=3.7" "pyyaml" "statsmodels>=0.14"
import transformers, ultralytics, albumentations
print("transformers", transformers.__version__, "| ultralytics", ultralytics.__version__)

## §3 · Unpack (native-res) and build the 640 cache once

Same setup as the training notebook: unzip the native-res package to local SSD and
build a 640-stretch cache **once**. SegFormer + the custom-YOLO path read the 640
cache; native YOLO reads the native-res images directly.

In [ ]:
import zipfile, cv2, numpy as np, pandas as pd

ZIP_SRC = Path(CFG["drive_dir"]) / CFG["zip_name"]
WORK    = Path(CFG["work_dir"])
if not ZIP_SRC.exists():
    raise FileNotFoundError(f"{ZIP_SRC} not found. Build with scripts/42 and upload.")
if ZIP_SRC.stat().st_size < 2e9:
    raise RuntimeError(f"{ZIP_SRC} is only {ZIP_SRC.stat().st_size/1e9:.2f} GB; expected ~2.7 GB native-res package.")

if not (WORK / "manifests" / "train.csv").exists():
    WORK.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(ZIP_SRC) as zf:
        zf.extractall(WORK)
    print(f"unzipped in {time.time()-t0:.0f}s")

MAN = {s: pd.read_csv(WORK / "manifests" / f"{s}.csv") for s in ("train", "val", "test")}
for s, df in MAN.items():
    print(f"{s:>5}: {len(df):>3} images, {df['subject'].nunique():>3} subjects")
assert (len(MAN["train"]), len(MAN["val"]), len(MAN["test"])) == (697, 134, 185)

RUNS_DIR = Path(CFG["drive_dir"]) / "runs_final"
OUT_DIR  = Path(CFG["drive_dir"]) / "results_final"
if not RUNS_DIR.exists():
    raise FileNotFoundError(f"{RUNS_DIR} not found -- run bruise_colab_final.ipynb first (checkpoints needed).")
print("checkpoints <-", RUNS_DIR)

In [ ]:
# Build the 640 stretch cache once (image bilinear, mask nearest -- albumentations'
# defaults, so this is bit-exact to what the training/eval dataloader computes live).
CACHE640 = WORK / "cache640"
def build_cache(df, split):
    idir = CACHE640 / split / "images"; mdir = CACHE640 / split / "masks"
    idir.mkdir(parents=True, exist_ok=True); mdir.mkdir(parents=True, exist_ok=True)
    for _, r in df.iterrows():
        ip = idir / f"{r.stem}.png"; mp = mdir / f"{r.stem}.png"
        if not ip.exists():
            im = cv2.imread(str(WORK / r.image_path), cv2.IMREAD_COLOR)
            cv2.imwrite(str(ip), cv2.resize(im, (640,640), interpolation=cv2.INTER_LINEAR))
        if not mp.exists():
            m = cv2.imread(str(WORK / r.mask_path), cv2.IMREAD_GRAYSCALE)
            if m.ndim == 3: m = m[..., 0]
            b = (m > 0).astype(np.uint8)
            cv2.imwrite(str(mp), cv2.resize(b, (640,640), interpolation=cv2.INTER_NEAREST) * 255)

if not (CACHE640 / "test" / "images").exists():
    t0 = time.time()
    for s in ("val","test"): build_cache(MAN[s], s)   # analysis only needs val (thr) + test
    print(f"640 cache built in {time.time()-t0:.0f}s")
else:
    print("640 cache present")

MAN640 = {}
for s in ("val","test"):
    d = MAN[s].copy()
    d["image_path"] = d["stem"].apply(lambda x: f"{s}/images/{x}.png")
    d["mask_path"]  = d["stem"].apply(lambda x: f"{s}/masks/{x}.png")
    MAN640[s] = d

## §4 · The library

`bruisekit` is reused **verbatim** from `bruise_colab_final.ipynb`, so inference
here is byte-identical to the run that produced `results_final/`. The mkdir comes
first so the `%%writefile` cells have a directory to land in.

In [ ]:
%cd /content
from pathlib import Path
Path("/content/bruisekit").mkdir(parents=True, exist_ok=True)
print("package dir ready:", Path("/content/bruisekit").is_dir())

In [ ]:
%%writefile bruisekit/__init__.py
"""Bruise segmentation training kit. See docs/training_notebook_explained.md."""

In [ ]:
%%writefile bruisekit/data.py
"""One dataloader for every model: one disk read, one resize, one augmentation.

THE DATASET EMITS RAW [0,1] PIXELS -- IT DOES NOT NORMALISE
------------------------------------------------------------
SegFormer wants ImageNet-normalised input; YOLO wants plain /255. That is not a
style preference, it is a property of the trained weights: Ultralytics trains on
/255 and its BatchNorms carry frozen running statistics for that distribution.
Feeding YOLO ImageNet-normalised pixels makes it under-fire by 4x and caps it at
Dice 0.479 with NO threshold able to recover it.

So pixel scale belongs to the MODEL, not the loader (see models.py). The loader
emits raw [0,1] and each wrapper applies its own scale. Every model therefore
shares one disk read, one resize and one augmentation -- the geometry is
identical by construction -- while each still sees the distribution it was
trained for. The alternative (normalise here, un-normalise for YOLO) works but
carries a needless roundtrip and invites exactly the bug it is working around.
"""
from __future__ import annotations

from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset


def build_augmentation(training: bool, img_size: int) -> A.Compose:
    """The augmentation pipeline. Identical for all five models.

    A.Resize is a NO-OP on the packaged 640x640 PNGs -- it is kept as a cheap
    guard so a wrong-sized file fails as a shape error here rather than as a
    silent geometry mismatch 40 minutes into training.

    A.Normalize(mean=0, std=1, max_pixel_value=255) is exactly x/255: albumentations
    computes (img - mean*max_pixel_value) / (std*max_pixel_value).
    """
    to_unit = A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0)
    resize = [A.Resize(height=img_size, width=img_size)]
    if not training:
        return A.Compose(resize + [to_unit, ToTensorV2()])
    return A.Compose(resize + [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
        A.GaussNoise(p=0.2),
        to_unit, ToTensorV2(),
    ])


class BruiseDataset(Dataset):
    """Reads the 640x640 PNG package. Returns (x[3,H,W] in [0,1], y[1,H,W] in {0,1}, stem)."""

    def __init__(self, df: pd.DataFrame, root: Path, img_size: int, training: bool = False):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.img_size = img_size
        self.tfm = build_augmentation(training, img_size)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        r = self.df.iloc[idx]

        img = cv2.imread(str(self.root / r.image_path), cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Cannot read image: {r.image_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(self.root / r.mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Cannot read mask: {r.mask_path}")
        # IMREAD_GRAYSCALE is documented to return (H, W) but returns (H, W, 1) in any
        # process where ultralytics has been imported -- it monkey-patches cv2.imread,
        # and import ORDER does not help. That trailing axis survives ToTensorV2 and
        # makes y [B,1,H,W,1]; dice(pred[H,W], gt[H,W,1]) then BROADCASTS to [H,W,H]
        # and returns nonsense -- a pixel-perfect prediction scores 63.9 instead of 1.0.
        # Squeezing here is a no-op on an unpatched cv2 and fixes every consumer at once.
        if mask.ndim == 3:
            mask = mask[..., 0]
        mask = (mask > 0).astype(np.float32)

        aug = self.tfm(image=img, mask=mask)
        x = aug["image"].float()
        y = aug["mask"].unsqueeze(0).float()
        assert y.shape == (1, self.img_size, self.img_size), f"bad mask shape {y.shape} for {r.stem}"
        return x, y, str(r.stem)


def make_loader(df, root, img_size, batch_size, training, workers, seed=0):
    """DataLoader with seeded, reproducible shuffling and worker RNG."""
    ds = BruiseDataset(df, root, img_size, training=training)
    gen = torch.Generator()
    gen.manual_seed(seed)

    def _init_worker(worker_id: int) -> None:
        # Each worker gets a distinct but seed-derived RNG stream, so augmentation
        # is reproducible for a given seed AND workers never draw identical noise.
        np.random.seed(seed * 1000 + worker_id)

    return torch.utils.data.DataLoader(
        ds, batch_size=batch_size, shuffle=training, drop_last=training,
        num_workers=workers, pin_memory=True,
        persistent_workers=workers > 0,
        worker_init_fn=_init_worker, generator=gen,
    )

In [ ]:
%%writefile bruisekit/models.py
"""The two architectures behind one interface.

THE INTERFACE
--------------
    forward_train(x) -> (logits[B,1,H,W], aux_logits[B,1,H,W] | None)
    forward(x)       -> logits[B,1,H,W]

x is RAW [0,1] pixels (see data.py). Each wrapper applies its own pixel scale.
Both models emit ONE bruise logit at full input resolution, so every downstream
consumer -- loss, sweep, metric, benchmark -- is architecture-blind. Nothing
below this line ever branches on a model's name.

WHY YOLO IS BUILT WITH nc=1 (not nc=2)
---------------------------------------
The pretrained yolo26n-sem.pt has nc=19 (Cityscapes). Rebuilding the head with
nc=2 gives [B,2,H,W], from which the bruise logit is z1-z0 (2-class softmax and
sigmoid-of-the-difference are the same function). Rebuilding with nc=1 gives that
single logit directly, and Ultralytics' own loss supports it (nn.BCEWithLogitsLoss
for nc==1). One less transformation, one less place to get the sign wrong, and
structurally identical to SegFormer's 1-channel head. `.load()` transfers 360/364
tensors -- only the head is new, exactly mirroring SegFormer's randomly-initialised
1-class head on a pretrained backbone.

WHY THE HEAD GETS 10x THE BACKBONE'S LR (both architectures)
-------------------------------------------------------------
The backbone is pretrained and already has good features; a conservative LR
preserves them. The head is randomly initialised and has to catch up. This is the
SegFormer paper's recipe (Xie et al. 2021) and it is applied to YOLO here too --
not because YOLO's paper says so, but because holding the recipe fixed across
architectures is the entire point of an apple-to-apple comparison.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class SegFormerNet(nn.Module):
    """HuggingFace SegFormer with a 1-class head. Input scale: ImageNet."""

    def __init__(self, pretrained_dir: str):
        super().__init__()
        from transformers import SegformerForSemanticSegmentation
        self.net = SegformerForSemanticSegmentation.from_pretrained(
            pretrained_dir, num_labels=1, ignore_mismatched_sizes=True)
        # Buffers (not constants) so they move with .to(device) and are saved with
        # the module -- the pixel scale travels with the weights it belongs to.
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    @property
    def backbone(self):
        return self.net.segformer

    @property
    def head(self):
        return self.net.decode_head

    def forward_train(self, x):
        x = (x - self.mean) / self.std                      # [0,1] -> ImageNet
        logits = self.net(pixel_values=x).logits            # [B,1,H/4,W/4]
        logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return logits, None                                 # SegFormer has no aux head

    def forward(self, x):
        return self.forward_train(x)[0]


class YoloSemNet(nn.Module):
    """Ultralytics YOLO26n-sem with an nc=1 head. Input scale: /255 (i.e. x as-is).

    In train() the underlying module returns (main[B,1,H/8,W/8], aux[B,1,H/16,W/16]);
    in eval() it returns only the main tensor. Both are upsampled to input
    resolution here so callers never see the stride.

    NOTE the stride: YOLO's main head is at H/8 (80x80 for a 640 input) against
    SegFormer's H/4 (160x160). YOLO predicts at a quarter of SegFormer's spatial
    resolution in area, which is an architectural ceiling on boundary precision,
    not something training can fix. Worth stating in the paper.
    """

    def __init__(self, pretrained_pt: str):
        super().__init__()
        from ultralytics import YOLO
        from ultralytics.nn.tasks import SemanticSegmentationModel

        pretrained = YOLO(str(pretrained_pt))
        self.net = SemanticSegmentationModel("yolo26n-sem.yaml", nc=1, ch=3, verbose=False)
        self.net.load(pretrained.model)     # transfers the backbone; head stays random
        del pretrained

    @property
    def backbone(self):
        return self.net.model[:-1]

    @property
    def head(self):
        return self.net.model[-1]

    def forward_train(self, x):
        out = self.net(x)                                   # x already /255
        if isinstance(out, (list, tuple)):
            main, aux = out[0], out[1]
        else:
            main, aux = out, None
        size = x.shape[-2:]
        main = F.interpolate(main.float(), size=size, mode="bilinear", align_corners=False)
        if aux is not None:
            aux = F.interpolate(aux.float(), size=size, mode="bilinear", align_corners=False)
        return main, aux

    def forward(self, x):
        return self.forward_train(x)[0]


def build_model(arch: str, size: str, paths: dict) -> nn.Module:
    if arch == "segformer":
        return SegFormerNet(paths[f"segformer_{size}"])
    if arch == "yolo":
        return YoloSemNet(paths["yolo"])
    raise ValueError(f"unknown arch: {arch}")


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def build_param_groups(model, backbone_lr: float, head_lr: float, weight_decay: float):
    """Backbone/head LR split + no weight decay on norms and biases.

    No decay on 1-D params (norm weights, biases): decaying a normalisation scale
    or a bias shrinks it toward zero with no regularising benefit -- standard in
    every transformer recipe including SegFormer's.

    Membership is by id(), not by name: the two architectures name their
    parameters completely differently, and a name-prefix rule would silently put
    every YOLO parameter in the wrong group.
    """
    backbone_ids = {id(p) for p in model.backbone.parameters()}
    groups = {
        "backbone_decay":    {"params": [], "lr": backbone_lr, "weight_decay": weight_decay},
        "backbone_no_decay": {"params": [], "lr": backbone_lr, "weight_decay": 0.0},
        "head_decay":        {"params": [], "lr": head_lr,     "weight_decay": weight_decay},
        "head_no_decay":     {"params": [], "lr": head_lr,     "weight_decay": 0.0},
    }
    for name, p in model.named_parameters():
        if not p.requires_grad or name in ("mean", "std"):
            continue
        where = "backbone" if id(p) in backbone_ids else "head"
        decay = "_no_decay" if (p.ndim <= 1 or "norm" in name.lower() or "bias" in name.lower()) else "_decay"
        groups[where + decay]["params"].append(p)

    out = [g for g in groups.values() if g["params"]]
    n_grouped = sum(len(g["params"]) for g in out)
    n_total = sum(1 for n, p in model.named_parameters() if p.requires_grad and n not in ("mean", "std"))
    assert n_grouped == n_total, f"param grouping lost {n_total - n_grouped} tensors"
    return out

In [ ]:
%%writefile bruisekit/losses.py
"""Losses. One supervised loss for every model; one distillation fusion.

WHY Dice+BCE
-------------
BCE alone is dominated by the background: bruises cover ~4.7% of pixels, so a
model predicting all-background already scores well on BCE and gets almost no
gradient toward the bruise. The Dice term is scale-invariant with respect to
object size and supplies gradient proportional to overlap, which is what we
actually report. Summing them is the standard combination for imbalanced medical
segmentation (Milletari et al. 2016 V-Net for Dice; Drozdzal et al. 2016 for the
combination), and it is also what Ultralytics' own SemanticSegmentationLoss does
(BCEWithLogits + binary Dice), so using it for both architectures does not
disadvantage YOLO relative to its native recipe.

WHAT THE DISTILLATION LOSS IS, AND WHAT IT IS NOT
--------------------------------------------------
    loss = alpha * DiceBCE(student_logits, GT)
         + (1 - alpha) * BCE(student_logits, sigmoid(teacher_logits / T_cal))

This is CALIBRATED SOFT-TARGET DISTILLATION, not Hinton et al. (2015) KD. The
difference matters for the paper and should not be papered over:

  - Hinton's KD divides BOTH the student's and the teacher's logits by a shared
    temperature T and multiplies the soft term by T^2 (to keep the soft gradient's
    magnitude comparable to the hard term's as T varies).
  - Here the student's logits are NOT temperature-scaled, and there is no T^2.
    T_cal is not a KD knob at all: it is the temperature fitted by NLL on val
    (Guo et al. 2017) that makes the teacher's probabilities CALIBRATED.

So the teacher is used as a better-calibrated estimate of P(bruise | pixel), and
the student regresses onto that probability. The justification is Menon et al.
(2021), "A statistical perspective on distillation": distillation helps to the
extent the teacher approximates the Bayes class-probability, which is exactly
what calibration improves. Do NOT cite Hinton for this formula as written.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class DiceBCELoss(nn.Module):
    """BCE + (1 - mean per-image soft Dice).

    Dice is computed PER IMAGE and then averaged, not pooled over the batch. A
    batch-pooled Dice lets one large bruise dominate the whole batch's gradient
    and lets an image with no predicted pixels hide inside a batch that scored
    well -- and per-image is what the reported metric does, so the loss and the
    metric agree.
    """

    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = (2 * inter + self.smooth) / (denom + self.smooth)
        return bce + (1.0 - dice.mean())


class SupervisedLoss(nn.Module):
    """DiceBCE on the main head + aux_weight * BCE on the auxiliary head.

    The aux term applies only to YOLO (SegFormer returns aux=None). 0.4 is
    Ultralytics' own weight for its semantic aux head; keeping their value means
    the aux head is supervised as its designers intended even though the rest of
    the recipe is ours.
    """

    def __init__(self, aux_weight: float = 0.4):
        super().__init__()
        self.main = DiceBCELoss()
        self.aux_weight = aux_weight

    def forward(self, logits, aux_logits, target):
        loss = self.main(logits, target)
        if aux_logits is not None:
            loss = loss + self.aux_weight * F.binary_cross_entropy_with_logits(aux_logits, target)
        return loss


class DistillLoss(nn.Module):
    """alpha * supervised(GT) + (1-alpha) * BCE(student, calibrated teacher prob).

    See the module docstring for what this is and is not.
    """

    def __init__(self, alpha: float = 0.5, aux_weight: float = 0.4):
        super().__init__()
        self.alpha = alpha
        self.sup = SupervisedLoss(aux_weight)

    def forward(self, logits, aux_logits, target, teacher_prob):
        hard = self.sup(logits, aux_logits, target)
        soft = F.binary_cross_entropy_with_logits(logits, teacher_prob)
        return self.alpha * hard + (1.0 - self.alpha) * soft

In [ ]:
%%writefile bruisekit/metrics.py
"""Metrics: Dice/IoU per image, and threshold-free AP for model selection."""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch


def dice_np(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2 * np.logical_and(pred, gt).sum() / denom)


def iou_np(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    union = np.logical_or(pred, gt).sum()
    return 1.0 if union == 0 else float(np.logical_and(pred, gt).sum() / union)


def compute_image_row(pred: np.ndarray, gt: np.ndarray, stem: str) -> dict:
    pred_b, gt_b = pred.astype(bool), gt.astype(bool)
    tp = int(np.logical_and(pred_b, gt_b).sum())
    fp = int(np.logical_and(pred_b, ~gt_b).sum())
    fn = int(np.logical_and(~pred_b, gt_b).sum())
    return {
        "stem": stem,
        "dice": dice_np(pred, gt), "iou": iou_np(pred, gt),
        "precision": 1.0 if tp + fp == 0 else tp / (tp + fp),
        "recall": 1.0 if tp + fn == 0 else tp / (tp + fn),
        "pred_positive_pixels": int(pred_b.sum()),
        "gt_positive_pixels": int(gt_b.sum()),
    }


def summarize(rows: list[dict]) -> dict:
    df = pd.DataFrame(rows)
    # "Complete miss" = the model output ZERO pixels on an image that has a bruise.
    # This is the metric that separates the models by more than label noise, and for
    # an injury-documentation tool it is the one that actually matters: a blank mask
    # is a missed injury. Reported as a first-class number, not buried in the tail.
    miss = (df["pred_positive_pixels"] == 0) & (df["gt_positive_pixels"] > 0)
    return {
        "n_images": int(len(df)),
        "mean_dice": float(df["dice"].mean()),
        "median_dice": float(df["dice"].median()),
        "mean_iou": float(df["iou"].mean()),
        "median_iou": float(df["iou"].median()),
        "mean_precision": float(df["precision"].mean()),
        "mean_recall": float(df["recall"].mean()),
        "complete_miss_count": int(miss.sum()),
        "complete_miss_rate": float(miss.mean()),
    }


class BinnedAP:
    """Threshold-free average precision over pixels, via probability histograms.

    WHY AP IS THE MODEL-SELECTION METRIC
    -------------------------------------
    The old pipeline saved best_model.pt by val Dice AT A FIXED 0.5 -- but the
    threshold is re-fitted afterwards anyway, and the fitted operating points are
    nowhere near 0.5 (YOLO's lands around 0.18). So 0.5-Dice selection asks "which
    epoch is best at an operating point we will not use?" and can pick the wrong
    epoch for any model whose calibration drifts during training. AP integrates
    over ALL thresholds, so the epoch choice cannot be biased by one arbitrary cut.

    WHY HISTOGRAMS AND NOT sklearn.average_precision_score
    -------------------------------------------------------
    134 val images x 640 x 640 = 55M pixels. Sorting 55M floats per epoch costs
    seconds of wall-clock and ~450 MB. Binning into 4096 buckets on the GPU makes
    the whole thing O(bins) in memory and effectively free, at a quantisation
    error of ~1/4096 in probability -- three orders of magnitude below the
    epoch-to-epoch differences it has to rank.
    """

    def __init__(self, bins: int = 4096, device: str = "cuda"):
        self.bins = bins
        self.pos = torch.zeros(bins, dtype=torch.float64, device=device)
        self.neg = torch.zeros(bins, dtype=torch.float64, device=device)

    @torch.no_grad()
    def update(self, prob: torch.Tensor, gt: torch.Tensor) -> None:
        p = prob.reshape(-1).float().clamp(0, 1)
        g = gt.reshape(-1) > 0.5
        idx = (p * (self.bins - 1)).round().long()
        self.pos += torch.bincount(idx[g], minlength=self.bins).double()
        self.neg += torch.bincount(idx[~g], minlength=self.bins).double()

    def compute(self) -> float:
        """AP = sum over thresholds of (recall_k - recall_{k-1}) * precision_k."""
        total_pos = self.pos.sum()
        if total_pos == 0:
            return float("nan")
        # Walk bins from the highest probability downward: each step admits one more
        # bucket as "predicted positive", which is exactly sweeping the threshold down.
        tp = torch.cumsum(self.pos.flip(0), dim=0)
        fp = torch.cumsum(self.neg.flip(0), dim=0)
        precision = tp / (tp + fp).clamp_min(1e-12)
        recall = tp / total_pos
        d_recall = torch.diff(recall, prepend=torch.zeros(1, dtype=recall.dtype, device=recall.device))
        return float((d_recall * precision).sum())

In [ ]:
%%writefile bruisekit/engine.py
"""The training loop. One loop for every model, and it survives session death."""
from __future__ import annotations

import json
import os
import random
import shutil
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

from bruisekit.data import make_loader
from bruisekit.losses import DistillLoss, SupervisedLoss
from bruisekit.metrics import BinnedAP
from bruisekit.models import build_model, build_param_groups, count_params


def seed_everything(seed: int) -> None:
    """Seed every RNG that touches training.

    cudnn.deterministic without use_deterministic_algorithms(True): the latter
    makes several ops here (bilinear interpolate backward, scatter_add) raise
    instead of run. Bitwise GPU determinism is therefore NOT claimed -- which is
    precisely why this notebook runs three seeds and reports spread rather than
    pretending one run is the answer.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def lr_multiplier(step: int, total_steps: int, warmup_steps: int, power: float = 1.0) -> float:
    """Linear warmup 0->1, then poly decay 1->0. SegFormer's schedule (Xie 2021)."""
    if step <= warmup_steps:
        return step / max(1, warmup_steps)
    progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
    return (1.0 - progress) ** power


def resolve_micro_batch(model, cfg, device, teacher=None) -> tuple[int, int]:
    """Choose (micro_batch, accum_steps) by probing what actually fits in VRAM.

    Two policies, selected by cfg["batch_mode"]:

    "matched" (the paper recipe) -- the probe is CAPPED AT effective_batch, and
    accumulation makes up any shortfall, so EVERY model trains at the same effective
    batch (8) and the same number of optimizer steps regardless of GPU. This is the
    fix for the old pipeline's central bug: the probe used to return 32 for the small
    models while accum collapsed to 1, so B2 trained at batch 8 / 8700 steps and both
    B0s at batch 32 / 2100 steps at the same LR -- making every "identical
    hyperparameters" claim between B0 and B2 false. A T4 just uses more accumulation
    than an A100 and lands in the same place.

    "per_model" -- the probe is NOT capped; each model uses the largest power-of-2
    batch its own size allows (accum = 1). B2 stays ~8, B0/YOLO climb much higher.
    Faster and better GPU utilisation, but the models NO LONGER share an effective
    batch or a step count, so this policy is not recipe-matched across model sizes
    (see the per-model notebook's §1/§6 warning). Same LR is kept deliberately (no
    linear-scaling adjustment), so the small models take fewer, larger steps.

    The probe runs on a DEEPCOPY: it does real forward/backward/step, and probing the
    live model would perturb the pretrained weights before training starts by a
    model-dependent amount.
    """
    mode = cfg.get("batch_mode", "matched")
    effective = cfg["effective_batch"]
    if not torch.cuda.is_available():
        # No GPU to probe: matched keeps its fixed effective batch via accumulation;
        # per_model has no target to hit, so fall back to a single sample.
        return (1, effective) if mode == "matched" else (1, 1)

    # matched caps the probe at effective_batch (accum fills the rest); per_model
    # lets it climb to max_probe_batch, whatever the model can hold.
    probe_ceiling = min(cfg["max_probe_batch"], effective) if mode == "matched" else cfg["max_probe_batch"]

    probe_model = deepcopy(model).to(device)
    probe_opt = torch.optim.SGD(probe_model.parameters(), lr=1e-9)
    scaler = torch.amp.GradScaler("cuda") if cfg["amp"] else None
    total_vram = torch.cuda.get_device_properties(device).total_memory

    chosen, batch = 1, 1
    while batch <= probe_ceiling:
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)
            x = torch.rand(batch, 3, cfg["img_size"], cfg["img_size"], device=device)
            y = torch.randint(0, 2, (batch, 1, cfg["img_size"], cfg["img_size"]), device=device).float()
            probe_model.train()
            probe_opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg["amp"]):
                # The teacher runs on every real training step too; excluding it here
                # would choose a batch that OOMs the moment distillation starts.
                if teacher is not None:
                    _ = teacher(x)
                logits, aux = probe_model.forward_train(x)
                loss = nn.functional.binary_cross_entropy_with_logits(logits, y)
                if aux is not None:
                    loss = loss + nn.functional.binary_cross_entropy_with_logits(aux, y)
            if scaler is not None:
                scaler.scale(loss).backward(); scaler.step(probe_opt); scaler.update()
            else:
                loss.backward(); probe_opt.step()
            frac = torch.cuda.max_memory_reserved(device) / total_vram
            del x, y, logits, loss
            # 0.75 leaves headroom for the real loss (Dice+BCE+aux), augmentation
            # variance and the checkpoint write -- the probe only sees a plain BCE.
            if frac > cfg.get("vram_target", 0.75):
                break
            chosen = batch
            batch *= 2
        except torch.cuda.OutOfMemoryError:
            break

    del probe_model, probe_opt
    torch.cuda.empty_cache()

    micro = max(1, chosen)
    if mode == "per_model":
        return micro, 1                    # use the probed batch directly, no accumulation
    while effective % micro != 0:          # matched: keep micro*accum EXACT, never approximate
        micro -= 1
    accum = effective // micro
    assert micro * accum == effective, f"{micro} x {accum} != {effective}"
    return micro, accum


def load_teacher(teacher_dir: Path, paths: dict, device, amp: bool):
    """Load a trained B2 as a frozen, calibrated soft-label generator.

    Returns a callable so the training loop never learns anything about the
    teacher's architecture -- it just calls teacher(x) and gets a probability map.
    """
    from bruisekit.models import build_model as _bm
    ckpt = teacher_dir / "best.pt"
    if not ckpt.exists():
        raise FileNotFoundError(f"Teacher not trained yet: {ckpt}")

    model = _bm("segformer", "b2", paths).to(device)
    model.load_state_dict(torch.load(str(ckpt), map_location=device, weights_only=True))
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    temperature = json.loads((teacher_dir / "calibration.json").read_text())["temperature"]

    def teacher_fn(x):
        # no_grad: the teacher is frozen. This is the single largest memory saving
        # in the distillation setup -- without it we would build a backward graph
        # through 27M parameters we never update.
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=amp):
            logits = model(x)
        return torch.sigmoid(logits.float() / temperature)

    teacher_fn.temperature = temperature
    return teacher_fn


def calibrate_temperature(model, loader, device, amp: bool) -> dict:
    """Fit the scalar T minimising val NLL of sigmoid(z/T). Guo et al. (2017).

    WHY: BCE drives a trained model's logits toward +-inf, because sigmoid(+-inf)
    is a perfect loss. The teacher's probability histogram ends up nearly binary,
    so sigmoid(z_teacher) as a soft label is almost indistinguishable from the hard
    GT -- which defeats the point of soft-label distillation. Dividing by T > 1
    pulls saturated logits back into sigmoid's responsive region, so the student
    can see where the teacher is UNCERTAIN, which is the information distillation
    is supposed to transfer.

    Optimises log(T), not T: keeps T > 0 automatically without a constraint, and
    avoids the singularity at T = 0.
    L-BFGS, not SGD: one scalar over a fixed dataset -- second-order curvature
    converges in ~10 iterations where SGD needs hundreds.
    """
    model.eval()
    logits_all, targets_all = [], []
    with torch.no_grad():
        for x, y, _ in tqdm(loader, desc="collect val logits", leave=False):
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=amp):
                z = model(x)
            # Downsample 4x before storing: calibration fits ONE scalar, and a
            # regular 1-in-16 pixel subsample of 55M pixels estimates it to far
            # more precision than it is worth -- while keeping this in RAM.
            logits_all.append(z.float()[..., ::4, ::4].cpu())
            targets_all.append(y[..., ::4, ::4].cpu())

    logits = torch.cat(logits_all)
    targets = torch.cat(targets_all)
    nll_before = float(torch.nn.functional.binary_cross_entropy_with_logits(logits, targets))

    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits / torch.exp(log_t), targets)
        loss.backward()
        return loss

    opt.step(closure)
    temperature = float(torch.exp(log_t).item())
    nll_after = float(torch.nn.functional.binary_cross_entropy_with_logits(logits / temperature, targets))

    # A well-trained BCE model is OVER-confident, so T lands a little above 1
    # (this project's B2 previously calibrated to 1.84). Anything far outside that
    # is a symptom, not a temperature:
    #   T >> 10  -> the logits are near zero, i.e. the model is barely trained and
    #               calibration is degenerately flattening an already-flat output.
    #               Distilling from it would teach the student a constant 0.5.
    #   T < 0.5  -> the model is UNDER-confident, which BCE training does not
    #               normally produce -- suspect the checkpoint or the val split.
    # Loud warning rather than a raise: the number is still recorded and the run can
    # proceed, but this must never pass silently into a paper.
    if not (0.5 <= temperature <= 10.0):
        print(f"  !! WARNING: calibrated T={temperature:.3f} is outside the plausible "
              f"[0.5, 10] range. The teacher is probably under-trained (near-zero "
              f"logits) -- check its val AP before trusting any distilled student.")

    return {"temperature": temperature, "nll_before": nll_before, "nll_after": nll_after,
            "n_pixels_used": int(targets.numel()), "plausible": bool(0.5 <= temperature <= 10.0)}


@torch.no_grad()
def eval_ap(model, loader, device, amp: bool) -> float:
    """Threshold-free val AP -- the model-selection metric. See metrics.BinnedAP."""
    model.eval()
    ap = BinnedAP(device=str(device))
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp):
            logits = model(x)
        ap.update(torch.sigmoid(logits.float()), y)
    return ap.compute()


def _atomic_save(obj, dest: Path) -> None:
    """Write to a temp file, then rename.

    Drive rename is atomic; a plain torch.save straight to Drive that is killed
    halfway leaves a TRUNCATED checkpoint, and the next session then dies trying
    to resume from it -- turning one lost session into a lost run.
    """
    tmp = dest.with_suffix(dest.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, dest)


def train_run(run_id: str, spec: dict, seed: int, cfg: dict, paths: dict,
              manifests: dict, root: Path, runs_dir: Path, device) -> dict:
    """Train one (model, seed). Idempotent and resumable.

    RESUME CONTRACT
    ----------------
    - DONE.json exists          -> return immediately, touch nothing.
    - resume.pt exists          -> restore model+optimizer+scaler+epoch+best and continue.
    - neither                   -> fresh start.

    resume.pt is written every cfg["drive_sync_every"] epochs (and always on the
    final epoch), so a disconnect costs at most that many epochs. It is saved even
    on epochs where val AP did NOT improve: the best weights live in best.pt, but
    resuming must continue from where training actually WAS, not from the last
    good epoch, or the LR schedule and optimizer moments silently rewind.
    """
    run_dir = runs_dir / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    done_file = run_dir / "DONE.json"
    if done_file.exists():
        return {"run_id": run_id, "status": "skipped", **json.loads(done_file.read_text())}

    seed_everything(seed)
    amp = cfg["amp"]

    model = build_model(spec["arch"], spec["size"], paths).to(device)

    teacher = None
    if spec["distill"]:
        # Same-seed teacher: the distilled run's spread over seeds then includes the
        # teacher's own variance, which is part of the pipeline being measured.
        teacher = load_teacher(runs_dir / f"segformer_b2_teacher__seed{seed}", paths, device, amp)

    micro, accum = resolve_micro_batch(model, cfg, device, teacher)

    train_loader = make_loader(manifests["train"], root, cfg["img_size"], micro,
                               training=True, workers=cfg["workers"], seed=seed)
    val_loader = make_loader(manifests["val"], root, cfg["img_size"], micro,
                             training=False, workers=cfg["workers"], seed=seed)

    param_groups = build_param_groups(model, cfg["backbone_lr"], cfg["head_lr"], cfg["weight_decay"])
    optimizer = torch.optim.AdamW(param_groups, betas=tuple(cfg["betas"]))
    peak_lrs = [g["lr"] for g in param_groups]
    scaler = torch.amp.GradScaler("cuda") if amp else None

    steps_per_epoch = max(1, len(train_loader) // accum)
    total_steps = steps_per_epoch * cfg["epochs"]
    warmup_steps = max(1, int(total_steps * cfg["warmup_fraction"]))

    criterion = (DistillLoss(cfg["alpha"], cfg["aux_weight"]) if teacher is not None
                 else SupervisedLoss(cfg["aux_weight"]))

    start_epoch, best_ap, patience, global_step, history = 1, float("-inf"), 0, 0, []
    resume_path = run_dir / "resume.pt"
    if resume_path.exists():
        st = torch.load(str(resume_path), map_location=device, weights_only=False)
        model.load_state_dict(st["model"])
        optimizer.load_state_dict(st["optimizer"])
        if scaler is not None and st.get("scaler"):
            scaler.load_state_dict(st["scaler"])
        start_epoch, best_ap = st["epoch"] + 1, st["best_ap"]
        patience, global_step, history = st["patience"], st["global_step"], st["history"]
        print(f"  [resume] {run_id} from epoch {start_epoch} (best_ap={best_ap:.4f})")
        del st

    (run_dir / "config.json").write_text(json.dumps({
        "run_id": run_id, "seed": seed, **spec,
        "micro_batch": micro, "accum_steps": accum, "effective_batch": micro * accum,
        "total_steps": total_steps, "warmup_steps": warmup_steps,
        "params": count_params(model),
        "alpha": cfg["alpha"] if spec["distill"] else None,
        "teacher_temperature": getattr(teacher, "temperature", None),
    }, indent=2))

    for epoch in range(start_epoch, cfg["epochs"] + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running, t0 = 0.0, time.time()

        for step, (x, y, _) in enumerate(tqdm(train_loader, desc=f"{run_id} e{epoch}", leave=False)):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=amp):
                # Teacher FIRST, inside its own no_grad: its activations are freed
                # before the student's backward graph is built. Student-first would
                # hold both alive at once and roughly double peak VRAM.
                tprob = teacher(x) if teacher is not None else None
                logits, aux = model.forward_train(x)
                loss = (criterion(logits, aux, y, tprob) if teacher is not None
                        else criterion(logits, aux, y))
                loss = loss / accum

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            running += loss.item() * accum

            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                global_step += 1
                # LR is set per OPTIMIZER STEP, not per micro-batch: the schedule is
                # defined over gradient updates, and updating it per micro-batch would
                # advance it accum_steps times too fast.
                mult = lr_multiplier(global_step, total_steps, warmup_steps, cfg["poly_power"])
                for g, peak in zip(optimizer.param_groups, peak_lrs):
                    g["lr"] = peak * mult
                if scaler is not None:
                    scaler.unscale_(optimizer)   # real gradients before clipping
                    nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        val_ap = eval_ap(model, val_loader, device, amp)
        train_loss = running / max(1, len(train_loader))
        cur_lr = peak_lrs[0] * lr_multiplier(global_step, total_steps, warmup_steps, cfg["poly_power"])
        history.append({"epoch": epoch, "train_loss": train_loss, "val_ap": val_ap,
                        "backbone_lr": cur_lr, "sec": round(time.time() - t0, 1)})
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

        if val_ap > best_ap:
            best_ap, patience = val_ap, 0
            _atomic_save(model.state_dict(), run_dir / "best.pt")
            flag = " *"
        else:
            patience += 1
            flag = ""
        print(f"  {run_id} e{epoch:3d} loss={train_loss:.4f} val_ap={val_ap:.4f}"
              f" lr={cur_lr:.2e} {time.time()-t0:.0f}s{flag}")

        last = (epoch == cfg["epochs"]) or (patience >= cfg["patience"])
        if epoch % cfg["drive_sync_every"] == 0 or last:
            _atomic_save({"epoch": epoch, "model": model.state_dict(),
                          "optimizer": optimizer.state_dict(),
                          "scaler": scaler.state_dict() if scaler else None,
                          "best_ap": best_ap, "patience": patience,
                          "global_step": global_step, "history": history}, resume_path)
        if patience >= cfg["patience"]:
            print(f"  early stop at epoch {epoch} (patience={cfg['patience']})")
            break

    # The teacher must be calibrated before any student can distil from it, and
    # calibration needs the BEST weights -- so it happens here, not in a separate step
    # that could be forgotten or run against the wrong checkpoint.
    model.load_state_dict(torch.load(str(run_dir / "best.pt"), map_location=device, weights_only=True))
    if run_id.startswith("segformer_b2_teacher"):
        cal = calibrate_temperature(model, val_loader, device, amp)
        (run_dir / "calibration.json").write_text(json.dumps(cal, indent=2))
        print(f"  calibrated T={cal['temperature']:.4f} "
              f"(val NLL {cal['nll_before']:.4f} -> {cal['nll_after']:.4f})")

    summary = {"run_id": run_id, "seed": seed, "best_val_ap": best_ap,
               "epochs_trained": len(history), "params": count_params(model),
               "micro_batch": micro, "accum_steps": accum}
    done_file.write_text(json.dumps(summary, indent=2))
    if resume_path.exists():
        resume_path.unlink()   # training finished: the resume state is now dead weight

    del model, teacher
    torch.cuda.empty_cache()
    return {"status": "trained", **summary}

In [ ]:
%%writefile bruisekit/sweep.py
"""Fit each model's operating point on val. Never on test."""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch


@torch.no_grad()
def cache_logits(model, loader, device, amp: bool):
    """One forward pass per image; keep the logits.

    The sweep tries 481 cuts. Re-running the model for each would be 481 x 134 =
    64,454 forward passes; caching the logits once makes all 481 cuts pure tensor
    arithmetic on data already on the GPU. fp16 keeps 134x640x640 at ~110 MB --
    the quantisation is ~3 decimal places on a logit, far below the resolution any
    of this can distinguish.
    """
    model.eval()
    logits, gts, stems = [], [], []
    for x, y, s in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp):
            z = model(x)
        logits.append(z.float().half()[:, 0])
        # GT as BOOL, never float: the sweep counts pixels, and counting is what
        # bool+int64 do exactly. See sweep_cuts for why that matters.
        gts.append((y[:, 0] > 0.5).to(device))
        stems.extend(s)
    return torch.cat(logits), torch.cat(gts), stems


def sweep_cuts(logits, gts, cuts):
    """Per-cut mean Dice, its standard error, and complete-miss rate. Vectorised.

    THE REDUCTIONS ARE EXACT, ON PURPOSE. Dice is a ratio of PIXEL COUNTS, so the
    intersection and the two cardinalities are computed as boolean masks summed to
    int64 -- exactly what metrics.dice_np does in numpy, and exact by construction.

    Doing the same reductions in fp16 (tempting, since the cached logits are fp16)
    is wrong: fp16 carries an 11-bit mantissa, and a 640x640 image sums ~410k terms,
    so the running total stops being able to represent +1 long before the sum
    finishes. Measured against the numpy implementation that drifted by ~1.5e-4 per
    cut -- small, but the tie band is selected by comparing cuts whose Dice differ
    by ~1e-3, so the error is a tenth of the signal it is used to rank. The logits
    stay fp16 (storage only; the >= comparison is exact regardless).
    """
    rows = []
    gts = gts.bool()
    gt_sum = gts.sum(dim=(1, 2))              # int64, exact
    gt_has = gt_sum > 0
    n = len(gt_sum)
    for c in cuts:
        pred = logits >= c                    # bool; comparison is exact in fp16
        inter = (pred & gts).sum(dim=(1, 2))
        pred_sum = pred.sum(dim=(1, 2))
        denom = pred_sum + gt_sum
        # denom == 0 means both prediction and GT are empty: a correct agreement,
        # scored 1.0 -- matching metrics.dice_np so the sweep and the report agree.
        dice = torch.where(denom > 0,
                           2.0 * inter.double() / denom.double().clamp_min(1.0),
                           torch.ones_like(denom, dtype=torch.float64))
        miss = ((pred_sum == 0) & gt_has).double()
        d = dice
        rows.append({
            "cut": float(c), "threshold": float(torch.sigmoid(torch.tensor(c))),
            "mean_dice": float(d.mean()),
            "se_dice": float(d.std(unbiased=True) / np.sqrt(n)),
            "complete_miss_rate": float(miss.mean()),
        })
    return pd.DataFrame(rows)


def select_cut(df: pd.DataFrame) -> dict:
    """Pick the operating point: the tie band's LOWEST-MISS cut.

    WHY A TIE BAND AT ALL
    ----------------------
    These sweeps are extraordinarily flat -- on the previous models, B2's val Dice
    moved by 0.009 across thresholds from 0.154 to 0.959. That is not a peak, it is
    noise on a plateau, and taking argmax of it fits the val set's sampling error.
    Every cut within ONE STANDARD ERROR of the peak is statistically tied, so the
    band -- not the argmax -- is the honest answer to "which cut is best?".

    WHY MISS RATE BREAKS THE TIE
    -----------------------------
    Cuts in the band are Dice-equivalent but they are NOT miss-equivalent: a lower
    cut predicts more pixels, so fewer images come back completely blank, at
    statistically identical Dice. The old rule took the band's MEDIAN cut, which
    optimises for stability -- but a blank mask is a missed injury, and the miss
    rate is the one metric that separates these models by more than label noise.
    So: minimise miss rate within the band; break remaining ties on Dice; break
    what is left on the median cut, for reproducibility.
    """
    peak = df.loc[df["mean_dice"].idxmax()]
    band = df[df["mean_dice"] >= peak["mean_dice"] - peak["se_dice"]]
    best_miss = band["complete_miss_rate"].min()
    tied = band[band["complete_miss_rate"] <= best_miss + 1e-12]
    top_dice = tied["mean_dice"].max()
    finalists = tied[tied["mean_dice"] >= top_dice - 1e-12]
    chosen = finalists.iloc[len(finalists) // 2]
    return {
        "cut": float(chosen["cut"]), "threshold": float(chosen["threshold"]),
        "val_dice_at_cut": float(chosen["mean_dice"]),
        "val_miss_at_cut": float(chosen["complete_miss_rate"]),
        "val_peak_dice": float(peak["mean_dice"]),
        "peak_cut": float(peak["cut"]),
        "band_lo_threshold": float(band["threshold"].min()),
        "band_hi_threshold": float(band["threshold"].max()),
        "band_width_cuts": int(len(band)),
        "n_cuts": int(len(df)),
    }

In [ ]:
%%writefile bruisekit/evaluate.py
"""Test scoring, fairness across skin tone, and the speed benchmark."""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch
from scipy import stats

from bruisekit.metrics import compute_image_row, summarize


@torch.no_grad()
def evaluate_at_cut(model, loader, device, cut: float, amp: bool) -> tuple[pd.DataFrame, dict]:
    """Score on test at an operating point ALREADY FIXED on val. One pass, no tuning."""
    model.eval()
    rows = []
    for x, y, stems in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp):
            z = model(x)
        pred = (z.float()[:, 0] >= cut).cpu().numpy().astype(np.uint8)
        gt = (y[:, 0] > 0.5).numpy().astype(np.uint8)
        for i, stem in enumerate(stems):
            rows.append(compute_image_row(pred[i], gt[i], stem))
    return pd.DataFrame(rows), summarize(rows)


def bootstrap_ci(values: np.ndarray, n: int = 2000, seed: int = 0) -> tuple[float, float]:
    """Percentile bootstrap CI of the MEDIAN.

    The median, not the mean: per-image Dice is strongly bimodal (a model either
    localises the bruise or misses it completely), so the mean mixes two different
    populations. And a bootstrap, not a t-interval, because that bimodality makes
    the normal approximation wrong.
    """
    if len(values) < 2:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    meds = [np.median(rng.choice(values, size=len(values), replace=True)) for _ in range(n)]
    return float(np.percentile(meds, 2.5)), float(np.percentile(meds, 97.5))


def fairness_analysis(per_image: pd.DataFrame, manifest: pd.DataFrame, model_name: str) -> dict:
    """Is performance equitable across Fitzpatrick/ITA skin-tone groups?

    THE STAKES: this is a forensic injury-documentation tool. A model that
    segments bruises well on light skin and poorly on dark skin does not have a
    metric problem, it has an evidentiary one -- it would under-document injuries
    on exactly the population most likely to need the documentation. So skin tone
    is not a robustness ablation here, it is a primary result.

    METHOD, and why each piece:
      - Groups are ITA (Individual Typology Angle, Chardon et al. 1991), the
        standard objective skin-tone measure -- computed from image pixels, not
        from a rater's Fitzpatrick guess, so it is reproducible.
      - Kruskal-Wallis across all 5 groups: an omnibus test for "does ANY group
        differ?". Non-parametric because per-image Dice is bimodal and bounded,
        so ANOVA's normality assumption fails.
      - Pairwise Mann-Whitney U, Bonferroni-corrected over the 10 pairs: with 5
        groups, uncorrected pairwise testing finds a "significant" pair ~40% of
        the time on pure noise.
      - fairness_gap = best group's median Dice - worst group's. The effect size.
        A p-value says a gap is real; only the gap says whether it matters.
    """
    df = per_image.merge(manifest[["stem", "skin_tone_category", "ita_group_index_5"]],
                         on="stem", how="left", validate="one_to_one")
    if df["ita_group_index_5"].isna().any():
        raise RuntimeError(f"{int(df['ita_group_index_5'].isna().sum())} test images have no skin-tone label")

    per_group, samples = [], []
    for gidx, g in sorted(df.groupby("ita_group_index_5"), key=lambda kv: kv[0]):
        vals = g["dice"].to_numpy()
        lo, hi = bootstrap_ci(vals)
        per_group.append({
            "model": model_name, "ita_group_index_5": int(gidx),
            "skin_tone_category": g["skin_tone_category"].iloc[0],
            "n_images": len(g), "median_dice": float(np.median(vals)),
            "iqr_dice": float(np.percentile(vals, 75) - np.percentile(vals, 25)),
            "ci95_lo": lo, "ci95_hi": hi,
            "mean_recall": float(g["recall"].mean()),
            "miss_rate": float(((g["pred_positive_pixels"] == 0) & (g["gt_positive_pixels"] > 0)).mean()),
        })
        samples.append(vals)

    H, p = stats.kruskal(*samples)
    pairwise = []
    raw = []
    pairs = [(i, j) for i in range(len(samples)) for j in range(i + 1, len(samples))]
    for i, j in pairs:
        raw.append(stats.mannwhitneyu(samples[i], samples[j], alternative="two-sided").pvalue)
    for (i, j), pv in zip(pairs, raw):
        adj = min(1.0, pv * len(pairs))    # Bonferroni over the 10 pairs
        pairwise.append({"model": model_name, "group_a": per_group[i]["skin_tone_category"],
                         "group_b": per_group[j]["skin_tone_category"],
                         "pvalue": pv, "bonferroni_p": adj, "significant": bool(adj < 0.05)})

    pg = pd.DataFrame(per_group)
    best, worst = pg.loc[pg["median_dice"].idxmax()], pg.loc[pg["median_dice"].idxmin()]
    stats_row = {
        "model": model_name, "kruskal_H": float(H), "kruskal_p": float(p),
        "significant": bool(p < 0.05),
        "fairness_gap": float(best["median_dice"] - worst["median_dice"]),
        "best_group": best["skin_tone_category"], "worst_group": worst["skin_tone_category"],
        "max_miss_rate_gap": float(pg["miss_rate"].max() - pg["miss_rate"].min()),
    }
    return {"per_group": pg, "pairwise": pd.DataFrame(pairwise), "stats": stats_row}


@torch.no_grad()
def benchmark_speed(model, images, device, cut: float, repeats: int, warmup: int) -> dict:
    """Time 640-tensor-on-GPU -> mask-on-GPU. Nothing else.

    WHAT IS DELIBERATELY NOT TIMED: disk read, JPEG decode, resize, host->GPU copy,
    and GPU->host copy. Those are identical for every model (one dataloader) and
    are dominated by I/O, so including them would compress the real architectural
    differences into measurement noise. The mask never leaves the GPU.

    cuda.synchronize() around each call is not optional: CUDA is asynchronous, so
    without it this would measure how long it takes to QUEUE the work, not do it --
    which reports every model as equally, impossibly fast.
    """
    model.eval()
    for _ in range(warmup):
        _ = torch.sigmoid(model(images[:1])) >= torch.sigmoid(torch.tensor(cut, device=device))
    torch.cuda.synchronize()

    times = []
    for _ in range(repeats):
        for i in range(len(images)):
            x = images[i:i + 1]
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            z = model(x)
            _ = z >= cut                      # threshold on the logit == sigmoid >= sigmoid(cut)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)

    arr = np.array(times)
    return {"median_ms": float(np.median(arr)), "mean_ms": float(arr.mean()),
            "p95_ms": float(np.percentile(arr, 95)), "fps": float(1000.0 / np.median(arr)),
            "n_timed": len(arr)}


import time  # noqa: E402  (used by benchmark_speed above)

In [ ]:
%%writefile bruisekit/postopt.py
"""Reduce complete misses WITHOUT retraining. Everything here is fitted on val and
applied to test; nothing changes a trained weight.

THE PROBLEM THIS SOLVES
------------------------
A single global threshold couples two failure modes: pushing it down to stop blank
masks (complete misses) also floods the easy images with false positives, so miss-%
and Dice move together -- two sides of one knob. Lowering the threshold slides ALONG
the miss-vs-Dice curve; it cannot move the curve. The three techniques here try to
move the curve (fewer misses at the SAME Dice), and one deliberately games the miss
metric so its cost is visible for comparison:

  ensemble   -- average the 3 seeds' probability maps. A miss needs ALL three seeds
                to blank the same image, which is rare (the per-seed misses are
                different images). Free: no retraining, just averaging maps we can
                already produce. This is the honest, recommended lever.
  TTA        -- average probs over horizontal+vertical flips. Raises the probability
                on borderline images so they clear the threshold without lowering it.
  no-blank   -- if a mask is still empty, recover the most-confident region instead of
                returning blank. This GAMES the miss metric (guarantees a non-zero
                prediction whether or not anything real was found), so it is reported
                as a separate floor, never as the main method.

HOW TO READ THE RESULT (this is the whole point of the question)
----------------------------------------------------------------
Plot miss-% against Dice. A real improvement sits BELOW-AND-LEFT of the single-model
threshold-sweep curve -- lower miss at equal-or-better Dice. A point that just slid
down the same curve (Dice fell as much as miss) is the threshold in disguise, not an
improvement. Everything below fits its threshold on VAL with the same miss-tie-break
rule as the baseline, so the comparison is like-for-like.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch

from bruisekit.metrics import compute_image_row, summarize
from bruisekit.sweep import select_cut


@torch.no_grad()
def probs_plain(model, loader, device, amp: bool):
    """One forward pass per image -> sigmoid probability maps [N,H,W] (fp16), GT
    (bool), stems. fp16 storage keeps the whole split in memory; the >= comparison
    the sweep does is exact regardless of storage dtype."""
    model.eval()
    P, G, S = [], [], []
    use_amp = amp and device.type == "cuda"
    for x, y, s in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            z = model(x)
        P.append(torch.sigmoid(z.float())[:, 0].half().cpu())
        G.append((y[:, 0] > 0.5).cpu())
        S.extend(s)
    return torch.cat(P), torch.cat(G), S


@torch.no_grad()
def probs_tta(model, loader, device, amp: bool, flips=("none", "h", "v")):
    """TTA: average sigmoid probs over identity + horizontal + vertical flip.

    Each flip is applied to the INPUT and UNDONE on the OUTPUT before averaging, so
    the maps stay pixel-aligned -- averaging misaligned maps would blur the boundary
    rather than sharpen the confidence. TTA is used identically on val (to fit the
    threshold) and test (to score); mismatching them would fit a threshold for a
    distribution the test pass never sees.
    """
    model.eval()
    P, G, S = [], [], []
    use_amp = amp and device.type == "cuda"
    for x, y, s in loader:
        x = x.to(device, non_blocking=True)
        acc = torch.zeros(x.shape[0], x.shape[2], x.shape[3], device=x.device)
        for f in flips:
            xf = torch.flip(x, [3]) if f == "h" else torch.flip(x, [2]) if f == "v" else x
            with torch.amp.autocast("cuda", enabled=use_amp):
                z = model(xf)
            p = torch.sigmoid(z.float())[:, 0]
            p = torch.flip(p, [2]) if f == "h" else torch.flip(p, [1]) if f == "v" else p
            acc = acc + p
        P.append((acc / len(flips)).half().cpu())
        G.append((y[:, 0] > 0.5).cpu())
        S.extend(s)
    return torch.cat(P), torch.cat(G), S


def mean_over_seeds(prob_list, stem_list):
    """Average probability maps across runs, aligned by stem.

    Loaders are shuffle=False so every seed already returns images in the same order,
    but this re-indexes by stem anyway rather than trusting that -- a silent order
    mismatch would average seed A's image 5 onto seed B's image 6 and quietly corrupt
    every ensemble number. Asserts the image SETS are identical first.
    """
    ref = stem_list[0]
    ref_set = set(ref)
    for sl in stem_list:
        if set(sl) != ref_set:
            raise ValueError("ensemble seeds cover different image sets")
    acc = None
    for probs, sl in zip(prob_list, stem_list):
        pos = {s: i for i, s in enumerate(sl)}
        reordered = torch.stack([probs[pos[s]] for s in ref]).float()
        acc = reordered if acc is None else acc + reordered
    return (acc / len(prob_list)).half(), list(ref)


def sweep_prob(probs, gts, thresholds):
    """Per-threshold mean Dice / SE / complete-miss on probability maps.

    Same exact-integer pixel counting as sweep.sweep_cuts (bool masks summed to
    int64), so the numbers are directly comparable to the logit-cut sweep. Emits the
    columns select_cut expects, with `cut` == `threshold` (probability space).

    The comparison is done in float32, NOT the fp16 the maps are stored in, so the
    threshold this sweep FITS on val is applied at the exact same numeric boundary
    score_prob_at() uses on test (it also compares in float32). Comparing fp16 here
    would put the val fit and the test apply on slightly different boundaries for any
    pixel sitting right at the threshold -- the same "the sweep must match the score"
    trap the logit sweep already guards against.
    """
    probs = probs.float()
    gts = gts.bool()
    gt_sum = gts.sum(dim=(1, 2))
    gt_has = gt_sum > 0
    n = len(gt_sum)
    rows = []
    for t in thresholds:
        pred = probs >= t
        inter = (pred & gts).sum(dim=(1, 2))
        ps = pred.sum(dim=(1, 2))
        den = ps + gt_sum
        dice = torch.where(den > 0, 2.0 * inter.double() / den.double().clamp_min(1.0),
                           torch.ones_like(den, dtype=torch.float64))
        miss = ((ps == 0) & gt_has).double()
        rows.append({"cut": float(t), "threshold": float(t),
                     "mean_dice": float(dice.mean()),
                     "se_dice": float(dice.std(unbiased=True) / np.sqrt(n)),
                     "complete_miss_rate": float(miss.mean())})
    return pd.DataFrame(rows)


def score_prob_at(probs, gts, stems, thr, no_blank=False, rel=0.5):
    """Score probability maps at a fixed threshold; optional no-blank floor.

    no_blank: when the thresholded mask is empty on an image that has a bruise, fall
    back to the region at >= rel * max-probability -- the most-confident blob the
    model saw. This never returns blank, which is exactly why it must be reported
    separately: it converts a genuine miss into a (possibly wrong) small prediction.
    """
    p_np = probs.float().numpy()
    g_np = gts.numpy()
    rows = []
    for i, s in enumerate(stems):
        p = p_np[i]
        pred = (p >= thr).astype("uint8")
        if no_blank and pred.sum() == 0 and p.max() > 0:
            pred = (p >= rel * float(p.max())).astype("uint8")
        rows.append(compute_image_row(pred, g_np[i].astype("uint8"), s))
    return pd.DataFrame(rows), summarize(rows)


def fit_on_val_apply_to_test(val_probs, val_gts, test_probs, test_gts, test_stems,
                             thresholds, no_blank=False):
    """The honest protocol: sweep val -> select_cut (miss-tie-break) -> score test.

    Returns (operating_point_dict, test_per_image_df, test_summary). The threshold is
    chosen ONLY from val, then applied once to test, exactly like the baseline in the
    main notebook -- so any miss/Dice difference is the technique, not a re-tuned cut.
    """
    grid = sweep_prob(val_probs, val_gts, thresholds)
    op = select_cut(grid)
    per_img, summ = score_prob_at(test_probs, test_gts, test_stems, op["threshold"], no_blank=no_blank)
    return op, per_img, summ

In [ ]:
%%writefile bruisekit/yolo_native.py
"""Native Ultralytics YOLO training + the two prediction paths.

This module deliberately uses Ultralytics (mosaic, EMA, letterbox, its own recipe) --
that is the whole point: YOLO is trained on its home turf, not on SegFormer's
transformer recipe. It is the ONLY module that imports ultralytics at train time.

TWO PREDICTION PATHS, and why both:
  native argmax   -- YOLO.predict() letterboxes to 640, argmaxes the 2-class head,
                     returns the mask at native resolution. We bring pred and GT to
                     640 (nearest) together and score. This is YOLO's best case and
                     reproduces its ~0.83 median.
  custom /255     -- pull the raw nn.Module, feed the SAME 640 stretch tensor SegFormer
                     sees (just /255, no ImageNet norm), take sigmoid(z1 - z0), and
                     sweep the threshold on val. Same geometry as SegFormer, so it is
                     the apples-to-apples-on-eval number.
"""
from __future__ import annotations

import copy
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from bruisekit.metrics import compute_image_row, summarize
from bruisekit.sweep import select_cut


def _ul_device() -> str:
    """Ultralytics device string: '0' on GPU (Colab), 'cpu' otherwise (local tests).

    Derived, not hardcoded, so the same module runs on the A100 in Colab and on a CPU
    box for verification without a code change -- a hardcoded '0' raises on CPU.
    """
    return "0" if torch.cuda.is_available() else "cpu"


def _read_native_mask(root: Path, rel: str) -> np.ndarray:
    m = cv2.imread(str(root / rel), cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise RuntimeError(f"Cannot read mask: {rel}")
    if m.ndim == 3:
        m = m[..., 0]
    return (m > 0).astype(np.uint8)


def _to_640(arr: np.ndarray) -> np.ndarray:
    return cv2.resize(arr.astype(np.uint8), (640, 640), interpolation=cv2.INTER_NEAREST)


# ── dataset build (native-res, Ultralytics semantic format) ──────────────────
def build_yolo_dataset(df, work_root: Path, out_dir: Path, split: str,
                       alpha=None, teacher_prob_fn=None, imgsz: int = 640) -> None:
    """images/<split>/ (symlink to native) + masks/<split>/ (0/1 class PNG, native res).

    For a distilled TRAIN split, the pseudo-mask fuses the teacher's native-resolution
    probability: class = (alpha*GT + (1-alpha)*teacher_prob >= 0.5). alpha < 0.5 keeps
    this non-degenerate (alpha > 0.5 would collapse it to plain GT). Val/test always use
    clean GT.
    """
    img_dir = out_dir / "images" / split; img_dir.mkdir(parents=True, exist_ok=True)
    msk_dir = out_dir / "masks" / split;  msk_dir.mkdir(parents=True, exist_ok=True)
    for _, r in df.iterrows():
        src = (work_root / r.image_path).resolve()
        dst = img_dir / Path(r.image_path).name
        if not dst.exists():
            try:
                dst.symlink_to(src)
            except OSError:
                shutil.copy2(src, dst)
        gt = _read_native_mask(work_root, r.mask_path).astype(np.float32)
        if split == "train" and teacher_prob_fn is not None:
            prob = teacher_prob_fn(work_root / r.image_path, gt.shape)
            cls = ((alpha * gt + (1.0 - alpha) * prob) >= 0.5).astype(np.uint8)
        else:
            cls = gt.astype(np.uint8)
        cv2.imwrite(str(msk_dir / f"{r.stem}.png"), cls)


def write_data_yaml(out_dir: Path, run_dir: Path) -> Path:
    import yaml
    data = {"path": str(out_dir.resolve()), "train": "images/train", "val": "images/val",
            "masks_dir": "masks", "names": {0: "background", 1: "bruise"}}
    p = run_dir / "data.yaml"
    with open(p, "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)
    return p


def train_native(weights_path: str, data_yaml: Path, run_dir: Path, cfg: dict, seed: int) -> Path:
    """Native Ultralytics training. Skips if best.pt exists; resumes from last.pt if
    interrupted (Ultralytics writes last.pt every epoch, so a disconnect costs <=1 epoch)."""
    from ultralytics import YOLO
    best = run_dir / "ultralytics_runs" / "train" / "weights" / "best.pt"
    last = run_dir / "ultralytics_runs" / "train" / "weights" / "last.pt"
    if best.exists():
        return best
    if last.exists():
        YOLO(str(last)).train(resume=True)
        return best
    YOLO(weights_path).train(
        data=str(data_yaml), task="semantic", imgsz=cfg["img_size"], epochs=cfg["epochs"],
        patience=cfg["patience"], batch=cfg["yolo_batch"], workers=cfg["workers"],
        device=_ul_device(), project=str(run_dir / "ultralytics_runs"), name="train", exist_ok=True,
        optimizer=cfg["yolo_optimizer"], lrf=cfg["yolo_lrf"], cos_lr=True,
        warmup_epochs=cfg["yolo_warmup_epochs"], weight_decay=cfg["yolo_weight_decay"],
        close_mosaic=cfg["yolo_close_mosaic"], seed=seed,
    )
    return best


# ── path (a): native argmax ──────────────────────────────────────────────────
def predict_native_argmax(best_pt: Path, df, work_root: Path, imgsz: int = 640):
    """YOLO.predict() argmax, pred+GT brought to 640 together and scored."""
    from ultralytics import YOLO
    model = YOLO(str(best_pt))
    rows = []
    for _, r in df.iterrows():
        res = model.predict(str(work_root / r.image_path), imgsz=imgsz, device=_ul_device(), verbose=False)[0]
        if getattr(res, "semantic_mask", None) is not None:
            cm = res.semantic_mask.data
            cm = cm.cpu().numpy() if hasattr(cm, "cpu") else np.asarray(cm)
            pred = (cm == 1).astype(np.uint8)
        else:
            pred = np.zeros((imgsz, imgsz), np.uint8)
        gt = _read_native_mask(work_root, r.mask_path)
        rows.append(compute_image_row(_to_640(pred), _to_640(gt), str(r.stem)))
    return pd.DataFrame(rows), summarize(rows)


# ── path (b): custom /255, raw module, val-swept threshold ───────────────────
def _raw_module(best_pt: Path, device):
    from ultralytics import YOLO
    return copy.deepcopy(YOLO(str(best_pt)).model).to(device).eval()


@torch.no_grad()
def _probs_640(model, df640, cache_root: Path, device):
    """sigmoid(z1 - z0) at 640, feeding the 640-stretch /255 tensor (SegFormer geometry)."""
    P, G, S = [], [], []
    for _, r in df640.iterrows():
        bgr = cv2.imread(str(cache_root / r.image_path), cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        x = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0).to(device)
        out = model(x)
        if isinstance(out, (list, tuple)):
            out = out[0]
        if out.shape[-2:] != (640, 640):
            out = F.interpolate(out.float(), size=(640, 640), mode="bilinear", align_corners=False)
        z = out[:, 1] - out[:, 0] if out.shape[1] >= 2 else out[:, 0]
        P.append(torch.sigmoid(z)[0].half().cpu())
        m = cv2.imread(str(cache_root / r.mask_path), cv2.IMREAD_GRAYSCALE)
        if m.ndim == 3: m = m[..., 0]
        G.append(torch.from_numpy((m > 0)))
        S.append(str(r.stem))
    return torch.stack(P), torch.stack(G), S


def predict_custom_255(best_pt: Path, val640, test640, cache_root: Path, device, thresholds):
    """Sweep the probability threshold on val, apply once to test. Same scoring as SegFormer."""
    from bruisekit.postopt import sweep_prob, score_prob_at
    model = _raw_module(best_pt, device)
    vp, vg, _ = _probs_640(model, val640, cache_root, device)
    grid = sweep_prob(vp, vg, thresholds)
    op = select_cut(grid)
    tp, tg, ts = _probs_640(model, test640, cache_root, device)
    per_img, summ = score_prob_at(tp, tg, ts, op["threshold"])
    return op, per_img, summ

## §5 · Imports, model paths, and the plotting style

In [ ]:
sys.path.insert(0, "/content")
import importlib
import bruisekit.data, bruisekit.models, bruisekit.metrics
import bruisekit.sweep, bruisekit.evaluate, bruisekit.yolo_native
for m in ("data","models","metrics","sweep","evaluate","yolo_native"):
    importlib.reload(sys.modules[f"bruisekit.{m}"])

from bruisekit.data import make_loader
from bruisekit.evaluate import evaluate_at_cut, fairness_analysis
from bruisekit.models import build_model, count_params
from bruisekit.sweep import cache_logits, select_cut, sweep_cuts
import bruisekit.yolo_native as yn

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr, kruskal, mannwhitneyu

PATHS = {
    "segformer_b0": str(WORK / "pretrained_weights" / "segformer_mit_b0"),
    "segformer_b2": str(WORK / "pretrained_weights" / "segformer_mit_b2"),
    "yolo":         str(WORK / "pretrained_weights" / "yolo26n-sem.pt"),
}
DEVICE = torch.device("cuda:0")

INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor":"#fcfcfb", "axes.facecolor":"#fcfcfb",
    "axes.edgecolor":"#c3c2b7", "axes.labelcolor":INK, "text.color":INK,
    "xtick.color":MUTED, "ytick.color":MUTED, "grid.color":GRID,
    "axes.spines.top":False, "axes.spines.right":False,
    "font.size":10, "figure.dpi":120,
})
print("style set; PATHS ->", {k: Path(v).name for k,v in PATHS.items()})

## §6 · Which seed, and one inference pass

We read the seed already chosen on **validation** from
`results_final/best_seed_val_selected/best_seed_val_selected_results.csv`. If that
file is missing (e.g. the folder was cleared) we re-derive it: score every seed on
val at its saved cut and take the best. Either way, only the winning seed is loaded
and predicted on test.

In [ ]:
BEST_CSV = OUT_DIR / "best_seed_val_selected" / "best_seed_val_selected_results.csv"

def _seg_val_dice(name, spec, seed):
    rd = RUNS_DIR / f"{name}__seed{seed}"
    if not (rd / "best.pt").exists() or not (rd / "operating_point.json").exists():
        return None
    model = build_model(spec["arch"], spec["size"], PATHS).to(DEVICE)
    model.load_state_dict(torch.load(str(rd/"best.pt"), map_location=DEVICE, weights_only=True))
    cut = json.loads((rd/"operating_point.json").read_text())["cut"]
    _, vs = evaluate_at_cut(model, make_loader(MAN640["val"], CACHE640, CFG["img_size"], 8, False, CFG["workers"], seed),
                            DEVICE, cut, CFG["amp"])
    del model; torch.cuda.empty_cache()
    return cut, vs["mean_dice"]

def _yolo_val_dice(name, seed):
    best = RUNS_DIR / f"{name}__seed{seed}" / "ultralytics_runs" / "train" / "weights" / "best.pt"
    if not best.exists():
        return None
    _, vs = yn.predict_native_argmax(best, MAN["val"], WORK, CFG["img_size"])
    return vs["mean_dice"]

BEST = {}   # name -> dict(seed, cut)
if BEST_CSV.exists():
    bt = pd.read_csv(BEST_CSV)
    for _, r in bt.iterrows():
        BEST[r["model"]] = {"seed": int(r["seed"]), "cut": float(r["cut"]) if pd.notna(r["cut"]) else None}
    print("best seeds read from disk:")
else:
    print("best_seed CSV absent -> re-deriving on val (slower)...")
    for name, spec in SEGFORMER_MODELS.items():
        cand = {s: _seg_val_dice(name, spec, s) for s in CFG["seeds"]}
        cand = {s: v for s, v in cand.items() if v is not None}
        bs = max(cand, key=lambda s: cand[s][1]); BEST[name] = {"seed": bs, "cut": cand[bs][0]}
    for name in YOLO_MODELS:
        cand = {s: _yolo_val_dice(name, s) for s in CFG["seeds"]}
        cand = {s: v for s, v in cand.items() if v is not None}
        bs = max(cand, key=lambda s: cand[s]); BEST[name] = {"seed": bs, "cut": None}
for k, v in BEST.items():
    print(f"  {k:<26} seed {v['seed']}  cut {v['cut']}")

In [ ]:
# One inference pass on the val-selected best seed. Produces:
#   per_image[name]              -> SegFormer / YOLO-native per-image rows (the CORE 5)
#   per_image_custom[name]       -> YOLO custom-/255 per-image rows (extra, for path comparison)
#   SEG_MODELS / YOLO_BEST       -> handles kept for the optional gallery (section G)
def enrich(df):
    """Add the derived columns the figures need (all algebra on the stored counts)."""
    df = df.copy()
    df["complete_miss"] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
    tp = (df.recall * df.gt_positive_pixels).round()
    df["tp_pixels"] = tp
    df["fp_pixels"] = (df.pred_positive_pixels - tp).clip(lower=0)
    df["fn_pixels"] = (df.gt_positive_pixels - tp).clip(lower=0)
    df["pred_gt_ratio"] = df.pred_positive_pixels / df.gt_positive_pixels.replace(0, np.nan)
    return df

per_image, per_image_custom = {}, {}
SEG_MODELS, YOLO_BEST = {}, {}

for name, spec in SEGFORMER_MODELS.items():
    seed, cut = BEST[name]["seed"], BEST[name]["cut"]
    rd = RUNS_DIR / f"{name}__seed{seed}"
    model = build_model(spec["arch"], spec["size"], PATHS).to(DEVICE)
    model.load_state_dict(torch.load(str(rd/"best.pt"), map_location=DEVICE, weights_only=True))
    pi, summ = evaluate_at_cut(model, make_loader(MAN640["test"], CACHE640, CFG["img_size"], 8, False, CFG["workers"], seed),
                              DEVICE, cut, CFG["amp"])
    per_image[name] = enrich(pi)
    SEG_MODELS[name] = (model.eval(), cut)   # kept on GPU for the gallery; A100 has room for all 3
    print(f"{DISP[name]:<26} seed {seed} | median {pi.dice.median():.3f}  miss {per_image[name].complete_miss.mean()*100:.2f}%")

for name in YOLO_MODELS:
    seed = BEST[name]["seed"]
    best = RUNS_DIR / f"{name}__seed{seed}" / "ultralytics_runs" / "train" / "weights" / "best.pt"
    YOLO_BEST[name] = best
    pi_a, _ = yn.predict_native_argmax(best, MAN["test"], WORK, CFG["img_size"])
    per_image[name] = enrich(pi_a)
    # custom /255 at the same seed (threshold swept on val) -- for the two-path comparison.
    PROB_THR = np.linspace(CFG["prob_min"], CFG["prob_max"], CFG["prob_steps"])
    _, pi_b, _ = yn.predict_custom_255(best, MAN640["val"], MAN640["test"], CACHE640, DEVICE, PROB_THR)
    per_image_custom[name] = enrich(pi_b)
    print(f"{DISP[name]:<26} seed {seed} | native median {pi_a.dice.median():.3f} miss {per_image[name].complete_miss.mean()*100:.2f}%"
          f" | custom median {pi_b.dice.median():.3f}")

# Subject + ITA labels come straight from the test manifest (no embedding, never stale).
SUBJ = MAN["test"][["stem","subject"]].copy()
ITA  = MAN["test"][["stem","skin_tone_category","ita_group_index_5"]].copy()
assert SUBJ.subject.nunique() == 28, SUBJ.subject.nunique()
print(f"\n{len(SUBJ)} images / {SUBJ.subject.nunique()} subjects | ITA groups: {sorted(ITA.skin_tone_category.unique())}")

In [ ]:
# Shared bootstrap helpers (subject-level cluster bootstrap; the 185 images are only
# 28 people, so we resample PEOPLE, not images). Paired where two models are compared.
RNG_SEED = 42
def _subject_index(df):
    subs = df["subject"].unique()
    return subs, {s: df[df.subject == s] for s in subs}

def cluster_ci(frame, stat, B=4000, seed=RNG_SEED):
    """95% CI of `stat(frame)` under subject resampling."""
    rng = np.random.default_rng(seed)
    subs, by = _subject_index(frame)
    vals = np.array([stat(pd.concat([by[s] for s in rng.choice(subs, len(subs), True)], ignore_index=True))
                     for _ in range(B)])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def paired_delta(frame, stat, B=4000, seed=RNG_SEED):
    """Point Δ, 95% CI, and one-sided P(Δ>0) for a paired statistic over subjects."""
    rng = np.random.default_rng(seed)
    subs, by = _subject_index(frame)
    point = stat(frame)
    vals = np.array([stat(pd.concat([by[s] for s in rng.choice(subs, len(subs), True)], ignore_index=True))
                     for _ in range(B)])
    return {"delta": float(point),
            "ci_lo": float(np.percentile(vals, 2.5)), "ci_hi": float(np.percentile(vals, 97.5)),
            "p_gt0": float((vals > 0).mean())}

def merged(a, b, cols=("dice",)):
    """Per-image join of two CORE variants on stem, with subject attached."""
    left  = per_image[a][["stem", *cols]].rename(columns={c: f"{c}_a" for c in cols})
    right = per_image[b][["stem", *cols]].rename(columns={c: f"{c}_b" for c in cols})
    return left.merge(right, on="stem").merge(SUBJ, on="stem")
print("bootstrap helpers ready")

---
# A · Accuracy & distribution

## A1 · Headline table (best seed) + on-disk 3-seed spread

Left: this best-seed run. Right (printed): the 3-seed mean±std already on disk in
`results_final/FINAL_RESULTS.csv`, so you can see the best seed sitting inside the
seed spread rather than cherry-picked outside it.

In [ ]:
rows = []
for name in CORE:
    d = per_image[name]
    rows.append({"variant": DISP[name], "median_dice": d.dice.median(), "mean_dice": d.dice.mean(),
                 "mean_iou": d.iou.mean(), "miss_%": d.complete_miss.mean()*100,
                 "seed": BEST[name]["seed"]})
head = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(head))[::-1]
ax.barh(y, head.mean_dice, height=0.6, color=[PALETTE[n] for n in CORE], zorder=3)
for yy, m, md_ in zip(y, head.mean_dice, head.median_dice):
    ax.text(m+0.006, yy, f"mean {m:.3f} / med {md_:.3f}", va="center", fontsize=8.5, color=INK)
ax.set_yticks(y); ax.set_yticklabels(head.variant, fontsize=9)
ax.set_xlim(0, 1.02); ax.set_xlabel("Dice (best seed, 185 test images)")
ax.set_title("Headline accuracy — best val-selected seed", fontsize=12, pad=10)
ax.grid(axis="x", lw=0.6, zorder=0); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

try:
    disk = pd.read_csv(OUT_DIR/"FINAL_RESULTS.csv")
    print("On-disk 3-seed mean±std (results_final/FINAL_RESULTS.csv):")
    print(disk[["variant","dice","median_dice","miss_%"]].to_string(index=False))
except Exception as e:
    print("FINAL_RESULTS.csv not read:", e)
head.round(4)

## A2 · Per-image Dice: violins + survival curves

The violin shows the spread over 185 images (white dot = median, red dash = mean).
The survival curve on the right, P(Dice ≥ x), makes the **worst-case tail** explicit:
where a curve drops early, that model has more low-Dice images — the failures a single
average number hides.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13.5, 4.6))
data = [per_image[n].dice.values for n in CORE]
parts = a1.violinplot(data, positions=np.arange(len(CORE)), widths=0.75,
                      showmeans=False, showmedians=False, showextrema=False)
for pc, n in zip(parts["bodies"], CORE):
    pc.set_facecolor(PALETTE[n]); pc.set_alpha(0.55); pc.set_edgecolor("#fcfcfb"); pc.set_linewidth(2)
for i, v in enumerate(data):
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    a1.vlines(i, q1, q3, color=INK, lw=5, zorder=3)
    a1.plot(i, med, "o", color="#fcfcfb", ms=6, zorder=4, markeredgecolor=INK, markeredgewidth=1.2)
    a1.plot(i, v.mean(), "_", color="#e34948", ms=16, mew=2.5, zorder=5)
a1.set_xticks(range(len(CORE))); a1.set_xticklabels([DISP[n] for n in CORE], rotation=20, ha="right", fontsize=7.5)
a1.set_ylabel("Per-image Dice"); a1.set_ylim(-0.02, 1.02)
a1.set_title("Distribution (white=median, red=mean)", fontsize=11); a1.grid(axis="y", lw=0.6); a1.set_axisbelow(True)

xs = np.linspace(0, 1, 200)
for n in CORE:
    v = per_image[n].dice.values
    surv = [(v >= x).mean() for x in xs]
    a2.plot(xs, surv, color=PALETTE[n], lw=2, label=DISP[n])
a2.set_xlabel("Dice threshold x"); a2.set_ylabel("P(Dice ≥ x)")
a2.set_title("Survival curves — the low-Dice tail", fontsize=11)
a2.grid(lw=0.6); a2.set_axisbelow(True)
a2.legend(fontsize=7, loc="lower left", frameon=False)
plt.tight_layout(); plt.show()
pd.DataFrame([{"model": DISP[n], "median": per_image[n].dice.median(), "mean": per_image[n].dice.mean(),
               "p25": per_image[n].dice.quantile(.25), "p05": per_image[n].dice.quantile(.05),
               "zeros": int((per_image[n].dice==0).sum())} for n in CORE]).round(4)

## A3 · Precision vs recall, and IoU vs Dice

Left: each dot is one image, precision vs recall. It shows an operating-point
character no single number does — e.g. YOLO tends to sit high-precision / lower-recall
(it fires *cleanly* but *misses*), which is exactly the under-detection the miss-rate
and fairness sections pick up. Right: IoU vs Dice, a consistency check (they should
track monotonically; the scatter is the per-image agreement).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13.5, 4.8))
for n in CORE:
    d = per_image[n]
    a1.scatter(d.recall, d.precision, s=14, alpha=0.45, color=PALETTE[n], edgecolors="none", label=DISP[n])
    a1.plot(d.recall.mean(), d.precision.mean(), "o", ms=11, color=PALETTE[n], markeredgecolor=INK, mew=1.2, zorder=5)
a1.set_xlabel("recall  (1 − under-segmentation)"); a1.set_ylabel("precision  (1 − over-segmentation)")
a1.set_xlim(0,1.02); a1.set_ylim(0,1.02); a1.set_title("Per-image precision vs recall (big dot = mean)", fontsize=11)
a1.grid(lw=0.6); a1.set_axisbelow(True); a1.legend(fontsize=7, loc="lower left", frameon=False)

for n in CORE:
    d = per_image[n]
    a2.scatter(d.dice, d.iou, s=13, alpha=0.4, color=PALETTE[n], edgecolors="none")
a2.plot([0,1],[0,1], color=MUTED, ls="--", lw=1)
a2.set_xlabel("Dice"); a2.set_ylabel("IoU"); a2.set_xlim(0,1.02); a2.set_ylim(0,1.02)
a2.set_title("IoU vs Dice (dashed = y=x)", fontsize=11); a2.grid(lw=0.6); a2.set_axisbelow(True)
plt.tight_layout(); plt.show()
pd.DataFrame([{"model": DISP[n], "mean_precision": per_image[n].precision.mean(),
               "mean_recall": per_image[n].recall.mean(), "mean_iou": per_image[n].iou.mean()}
              for n in CORE]).round(4)

---
# B · Safety & failure modes

## B1 · Complete-miss rate — the failure that matters, and "best Dice ≠ safest"

A **complete miss** = a real bruise, empty predicted mask. For an injury-documentation
tool that is the decisive failure: a loose outline is correctable, a blank mask is
silence about an injury that is present. The right panel plots miss vs median Dice —
watch the model with the best Dice not being the safest.

In [ ]:
rows = []
for n in CORE:
    f = per_image[n][["stem","complete_miss"]].merge(SUBJ, on="stem")
    f["complete_miss"] = f.complete_miss.astype(float)
    lo, hi = cluster_ci(f, lambda x: x.complete_miss.mean()*100, B=3000)
    rows.append({"model": n, "miss_pct": f.complete_miss.mean()*100, "lo": lo, "hi": hi,
                 "median_dice": per_image[n].dice.median()})
miss = pd.DataFrame(rows)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
xs = np.arange(len(miss))
a1.bar(xs, miss.miss_pct, width=0.55, color=[PALETTE[n] for n in miss.model], zorder=3)
a1.errorbar(xs, miss.miss_pct, yerr=[miss.miss_pct-miss.lo, miss.hi-miss.miss_pct],
            fmt="none", ecolor=INK, elinewidth=1.5, capsize=5, zorder=4)
for x, v, hi in zip(xs, miss.miss_pct, miss.hi):
    a1.text(x, hi+0.3, f"{v:.1f}%", ha="center", fontsize=9, color=INK)
a1.set_xticks(xs); a1.set_xticklabels([DISP[n] for n in miss.model], rotation=20, ha="right", fontsize=7.5)
a1.set_ylabel("Complete-miss rate (%)"); a1.set_title("Blank-mask failures (lower better)", fontsize=11)
a1.grid(axis="y", lw=0.6, zorder=0); a1.set_axisbelow(True)
for _, r in miss.iterrows():
    a2.scatter(r.miss_pct, r.median_dice, s=150, color=PALETTE[r.model], edgecolors=INK, lw=0.6, zorder=3)
    a2.annotate(DISP[r.model], (r.miss_pct, r.median_dice), fontsize=7.5, textcoords="offset points", xytext=(7,5))
a2.set_xlabel("Complete-miss rate (%)  →  worse"); a2.set_ylabel("Median Dice  →  better")
a2.set_title("Best Dice ≠ safest model", fontsize=11); a2.grid(lw=0.6, zorder=0); a2.set_axisbelow(True)
plt.tight_layout(); plt.show()
miss.assign(model=lambda d: d.model.map(DISP)).round(3)

## B2 · Over- vs under-segmentation — *how* they fail

`pred / GT area` = predicted bruise area ÷ true area. **<1** under-segments (misses
bruise — bad for evidence); **>1** over-segments (flags healthy skin — recoverable).
Which way a model leans matters more forensically than its total error.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.4))
rng = np.random.default_rng(0)
for i, n in enumerate(CORE):
    v = per_image[n].pred_gt_ratio.replace([np.inf,-np.inf], np.nan).dropna().clip(0, 3)
    ax.scatter(rng.normal(i, 0.07, len(v)), v, s=13, alpha=0.5, color=PALETTE[n], edgecolors="none", zorder=3)
    ax.plot(i, v.median(), "_", color=INK, ms=28, mew=2.5, zorder=4)
ax.axhline(1.0, color="#e34948", ls="--", lw=1.4, zorder=2)
ax.text(len(CORE)-0.45, 1.04, "perfect size", color="#e34948", fontsize=8.5)
ax.set_xticks(range(len(CORE))); ax.set_xticklabels([DISP[n] for n in CORE], rotation=20, ha="right", fontsize=7.5)
ax.set_ylabel("pred / GT area (clipped at 3)")
ax.set_title("Under- (<1) vs over- (>1) segmentation; black dash = median", fontsize=11.5, pad=10)
ax.grid(axis="y", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
pd.DataFrame([{"model": DISP[n], "median_pred_gt_ratio": per_image[n].pred_gt_ratio.median(),
               "under_seg_%": (per_image[n].pred_gt_ratio < 1).mean()*100,
               "mean_FP_px": per_image[n].fp_pixels.mean(), "mean_FN_px": per_image[n].fn_pixels.mean()}
              for n in CORE]).round(2)

## B3 · YOLO: native argmax vs custom /255 — which path, and where they differ

YOLO is evaluated two ways: **native argmax** (its home turf, letterbox + argmax) and
**custom /255** (the same 640 geometry SegFormer sees, threshold swept on val). Same
weights, different eval. The paired per-image delta shows the paths are not
interchangeable — report both and say which is which.

In [ ]:
fig, axes = plt.subplots(1, len(YOLO_MODELS), figsize=(11, 4.4))
comp_rows = []
for ax, n in zip(np.atleast_1d(axes), YOLO_MODELS):
    m = per_image[n][["stem","dice"]].rename(columns={"dice":"native"}).merge(
        per_image_custom[n][["stem","dice"]].rename(columns={"dice":"custom"}), on="stem")
    ax.scatter(m.native, m.custom, s=16, alpha=0.5, color=PALETTE[n], edgecolors="none")
    ax.plot([0,1],[0,1], color=MUTED, ls="--", lw=1)
    ax.set_xlabel("native argmax Dice"); ax.set_ylabel("custom /255 Dice")
    ax.set_xlim(0,1.02); ax.set_ylim(0,1.02); ax.set_title(DISP[n], fontsize=10)
    ax.grid(lw=0.6); ax.set_axisbelow(True)
    comp_rows.append({"model": DISP[n], "native_median": per_image[n].dice.median(),
                      "custom_median": per_image_custom[n].dice.median(),
                      "native_miss_%": per_image[n].complete_miss.mean()*100,
                      "custom_miss_%": per_image_custom[n].complete_miss.mean()*100,
                      "mean_abs_delta": (m.native-m.custom).abs().mean()})
fig.suptitle("YOLO evaluation paths are not interchangeable (dashed = agreement)", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
pd.DataFrame(comp_rows).round(4)

---
# C · Inference statistics (which differences are real?)

The 185 images come from only **28 people**, so photos of one person are not
independent. Every interval below is a **subject-level cluster bootstrap** (resample
people, B=4000); model-vs-model contrasts are **paired** (the same resample scores
both models — correct for a shared test set and ~2× tighter than unpaired).

## C1 · Marginal CIs — mean Dice per model

In [ ]:
rows = []
for n in CORE:
    f = per_image[n][["stem","dice"]].merge(SUBJ, on="stem")
    lo, hi = cluster_ci(f, lambda x: x.dice.mean(), B=4000)
    rows.append({"model": n, "mean": f.dice.mean(), "lo": lo, "hi": hi})
ci = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8.5, 4))
y = np.arange(len(ci))[::-1]
for yy, (_, r) in zip(y, ci.iterrows()):
    ax.plot([r.lo, r.hi], [yy, yy], color=PALETTE[r.model], lw=3, solid_capstyle="round", zorder=3)
    ax.plot(r["mean"], yy, "o", ms=9, color=PALETTE[r.model], markeredgecolor="#fcfcfb", mew=1.5, zorder=4)
    ax.text(r.hi+0.004, yy, f"{r['mean']:.3f} [{r.lo:.3f}, {r.hi:.3f}]", va="center", fontsize=8.5)
ax.set_yticks(y); ax.set_yticklabels([DISP[n] for n in ci.model], fontsize=9)
ax.set_xlabel("Mean Dice (95% subject cluster-bootstrap)"); ax.set_xlim(0.55, 0.92)
ax.set_title("Overlapping intervals: at 28 subjects the models are hard to separate", fontsize=11, pad=10)
ax.grid(axis="x", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
ci.assign(model=lambda d: d.model.map(DISP)).round(4)

## C2 · Paired contrasts — forest plot with P(Δ>0)

Each row is one paired comparison: Δ mean Dice with its 95% CI and the one-sided
**P(Δ>0)** (a 92% and a 52% are both "n.s." but mean very different things). A
contrast is resolvable only if its interval clears zero.

In [ ]:
CONTRASTS = [
    ("Distillation (B0-dist − B0-direct)", "segformer_b0_distilled", "segformer_b0_direct", "dice"),
    ("Student − teacher (B0-dist − B2)",   "segformer_b0_distilled", "segformer_b2_teacher", "dice"),
    ("SegFormer − YOLO (B0-dist − YOLO-d)","segformer_b0_distilled", "yolo_sem_direct",      "dice"),
    ("YOLO distill − direct",              "yolo_sem_distilled",     "yolo_sem_direct",      "dice"),
]
res = []
for label, a, b, col in CONTRASTS:
    m = merged(a, b, cols=(col,))
    r = paired_delta(m, lambda x: x[f"{col}_a"].mean() - x[f"{col}_b"].mean(), B=4000)
    r["label"] = label; res.append(r)
# add the miss-rate contrast that historically IS significant
mm = per_image["yolo_sem_direct"][["stem","complete_miss"]].rename(columns={"complete_miss":"a"}).merge(
     per_image["segformer_b0_distilled"][["stem","complete_miss"]].rename(columns={"complete_miss":"b"}), on="stem").merge(SUBJ,on="stem")
mm["a"]=mm.a.astype(float); mm["b"]=mm.b.astype(float)
rm = paired_delta(mm, lambda x: (x.a.mean()-x.b.mean())*100, B=4000); rm["label"]="MISS %: YOLO-direct − B0-dist"
FOREST = pd.DataFrame(res + [rm])

fig, ax = plt.subplots(figsize=(9.5, 4.4))
y = np.arange(len(FOREST))[::-1]
for yy, (_, r) in zip(y, FOREST.iterrows()):
    sig = r.ci_lo > 0 or r.ci_hi < 0
    c = "#c0392b" if sig else MUTED
    ax.plot([r.ci_lo, r.ci_hi], [yy, yy], color=c, lw=3, solid_capstyle="round", zorder=3)
    ax.plot(r.delta, yy, "o", ms=8, color=c, zorder=4)
    ax.text(r.ci_hi+0.002 if r.ci_hi>=0 else r.ci_hi, yy+0.28,
            f"Δ={r.delta:+.3f}  P(Δ>0)={r.p_gt0*100:.0f}%", fontsize=8, color=INK)
ax.axvline(0, color=INK, lw=1.1, zorder=2)
ax.set_yticks(y); ax.set_yticklabels(FOREST.label, fontsize=8.5)
ax.set_xlabel("Δ (paired subject bootstrap; Dice contrasts, except the miss-% row in %)")
ax.set_title("Paired effect sizes — red = interval clears zero", fontsize=11, pad=10)
ax.grid(axis="x", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
FOREST[["label","delta","ci_lo","ci_hi","p_gt0"]].round(4)

## C3 · Per-subject heatmap + model agreement

Left: each of the 28 subjects × each model, mean Dice (darker = better; hardest
subjects on top). Rows dark across *every* column are hard **subjects**, not model
failures — and with only 28 rows, one or two of them move the whole average (why C1's
bars are wide). Right: Spearman correlation of per-image Dice between models — high
everywhere means the models agree on which images are hard, so an ensemble helps less
than you'd hope.

In [ ]:
mats = []
for n in CORE:
    s = per_image[n][["stem","dice"]].merge(SUBJ, on="stem").groupby("subject").dice.mean()
    mats.append(s.rename(DISP[n]))
H = pd.concat(mats, axis=1)
H = H.loc[H.mean(axis=1).sort_values().index]
D = pd.DataFrame({DISP[n]: per_image[n].set_index("stem").dice for n in CORE})
C = D.corr(method="spearman")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 8), gridspec_kw={"width_ratios":[1, 1.05]})
im1 = a1.imshow(H.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
a1.set_xticks(range(len(H.columns))); a1.set_xticklabels(H.columns, rotation=35, ha="right", fontsize=7.5)
a1.set_yticks(range(len(H))); a1.set_yticklabels(H.index, fontsize=7)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        a1.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                color="#ffffff" if v > 0.55 else INK)
a1.set_title("Per-subject mean Dice (hardest on top)", fontsize=11)
fig.colorbar(im1, ax=a1, shrink=0.5, label="Mean Dice")

im2 = a2.imshow(C.values, cmap="Blues", vmin=0, vmax=1)
a2.set_xticks(range(len(C))); a2.set_xticklabels(C.columns, rotation=35, ha="right", fontsize=7.5)
a2.set_yticks(range(len(C))); a2.set_yticklabels(C.index, fontsize=7.5)
for i in range(len(C)):
    for j in range(len(C)):
        a2.text(j, i, f"{C.values[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="#ffffff" if C.values[i,j] > 0.55 else INK)
a2.set_title("Spearman corr. of per-image Dice", fontsize=11)
fig.colorbar(im2, ax=a2, shrink=0.7, label="ρ")
plt.tight_layout(); plt.show()
print("Hardest 5 subjects (mean over models):")
print(H.mean(axis=1).head(5).round(3).to_string())

---
# D · ⭐ Fairness across skin tone (ITA)

This is a forensic tool: a model that documents bruises well on light skin and poorly
on dark skin has an **evidentiary** problem, not a metric one. Groups are ITA
(Individual Typology Angle) — an objective, pixel-computed skin-tone measure. Computed
here on the **best-seed** per-image data via the same `fairness_analysis` used to make
`results_final/fairness_*.csv` (which are 3-seed-averaged).

**Honest caveat up front:** each ITA group holds only ~9–17 *subjects*. The
group-level CIs are wide and mostly overlap — treat these as **exploratory**. The
gap's *sign* is a hypothesis, not a proven effect, at n=28.

In [ ]:
# Fairness fresh from best-seed per-image, for the 5 core variants + both YOLO paths.
fair_frames = {}
for n in SEGFORMER_MODELS:
    fair_frames[DISP[n]] = per_image[n]
for n in YOLO_MODELS:
    fair_frames[f"{DISP[n]} · native"] = per_image[n]
    fair_frames[f"{DISP[n]} · custom"] = per_image_custom[n]

fg, fp, fs = [], [], []
for label, pi in fair_frames.items():
    out = fairness_analysis(pi[["stem","dice","recall","pred_positive_pixels","gt_positive_pixels"]],
                            MAN["test"], label)
    fg.append(out["per_group"]); fp.append(out["pairwise"]); fs.append(out["stats"])
FAIR_GROUP = pd.concat(fg, ignore_index=True)
FAIR_PAIR  = pd.concat(fp, ignore_index=True)
FAIR_STATS = pd.DataFrame(fs)
print(FAIR_STATS[["model","kruskal_p","significant","fairness_gap","best_group","worst_group","max_miss_rate_gap"]].to_string(index=False))

## D1 · Per-group heatmaps — median Dice, recall, miss-rate

In [ ]:
GROUP_ORDER = ["Light (II-III)","Intermediate (III-IV)","Tan (IV)","Brown (V)","Dark (VI)"]
def heat(ax, values, title, cmap, fmt, vmax):
    piv = FAIR_GROUP.pivot_table(index="model", columns="skin_tone_category", values=values)
    piv = piv.reindex(columns=[g for g in GROUP_ORDER if g in piv.columns])
    im = ax.imshow(piv.values, cmap=cmap, aspect="auto", vmin=0, vmax=vmax)
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=30, ha="right", fontsize=7)
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=7)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, fmt(piv.values[i,j]), ha="center", va="center", fontsize=6.5,
                    color="#ffffff" if (piv.values[i,j]/vmax) > 0.55 else INK)
    ax.set_title(title, fontsize=10)
    return im

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
i1 = heat(axes[0], "median_dice", "Median Dice", "Blues", lambda v: f"{v:.2f}", 1.0)
i2 = heat(axes[1], "mean_recall", "Mean recall", "Greens", lambda v: f"{v:.2f}", 1.0)
fair_miss = FAIR_GROUP.assign(miss_pct=FAIR_GROUP.miss_rate*100)
piv = fair_miss.pivot_table(index="model", columns="skin_tone_category", values="miss_pct").reindex(columns=[g for g in GROUP_ORDER if g in fair_miss.skin_tone_category.unique()])
im3 = axes[2].imshow(piv.values, cmap="Reds", aspect="auto", vmin=0, vmax=max(1e-6, np.nanmax(piv.values)))
axes[2].set_xticks(range(piv.shape[1])); axes[2].set_xticklabels(piv.columns, rotation=30, ha="right", fontsize=7)
axes[2].set_yticks(range(piv.shape[0])); axes[2].set_yticklabels(piv.index, fontsize=7)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        axes[2].text(j, i, f"{piv.values[i,j]:.0f}", ha="center", va="center", fontsize=6.5, color=INK)
axes[2].set_title("Complete-miss rate (%)", fontsize=10)
for im, ax in [(i1,axes[0]),(i2,axes[1]),(im3,axes[2])]:
    fig.colorbar(im, ax=ax, shrink=0.6)
fig.suptitle("Performance by ITA skin-tone group (rows = variants)", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
FAIR_GROUP.pivot_table(index="model", columns="skin_tone_category", values="median_dice").round(3)

## D2 · Fairness gap + the light-skin under-detection story

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6))
# gap bars, coloured by omnibus significance
order = FAIR_STATS.sort_values("fairness_gap")
xs = np.arange(len(order))
cols = ["#c0392b" if s else MUTED for s in order.significant]
a1.barh(xs, order.fairness_gap, color=cols, zorder=3)
for x, g, p in zip(xs, order.fairness_gap, order.kruskal_p):
    a1.text(g+0.003, x, f"{g:.3f} (KW p={p:.2f})", va="center", fontsize=7.5)
a1.set_yticks(xs); a1.set_yticklabels(order.model, fontsize=7.5)
a1.set_xlabel("fairness gap = best − worst group median Dice")
a1.set_title("Max−min Dice gap (red = Kruskal p<0.05)", fontsize=10.5)
a1.grid(axis="x", lw=0.6); a1.set_axisbelow(True)

# precision vs recall per group for one YOLO path -> under-detection vs confusion
yl = f"{DISP['yolo_sem_direct']} · native"
sub = FAIR_GROUP[FAIR_GROUP.model == yl]
# recompute per-group precision on the fly for this panel
pi = per_image["yolo_sem_direct"].merge(ITA, on="stem")
grp = pi.groupby("skin_tone_category").agg(recall=("recall","mean"), precision=("precision","mean"),
                                           miss=("complete_miss","mean")).reindex([g for g in GROUP_ORDER if g in pi.skin_tone_category.unique()])
a2.scatter(grp.recall, grp.precision, s=140, c=range(len(grp)), cmap="viridis", edgecolors=INK, zorder=3)
for g, r in grp.iterrows():
    a2.annotate(g.split(" ")[0], (r.recall, r.precision), fontsize=7.5, textcoords="offset points", xytext=(6,4))
a2.set_xlabel("recall"); a2.set_ylabel("precision")
a2.set_title(f"{DISP['yolo_sem_direct']}: per-group precision vs recall\n(low recall + high precision = under-detection)", fontsize=9.5)
a2.grid(lw=0.6); a2.set_axisbelow(True)
plt.tight_layout(); plt.show()
grp.assign(miss_pct=lambda d: d.miss*100).round(3)

## D3 · Pairwise significance (Bonferroni) — mostly n.s. at n=28

In [ ]:
sig = FAIR_PAIR.groupby("model").significant.sum()
tot = FAIR_PAIR.groupby("model").significant.count()
tab = pd.DataFrame({"significant_pairs": sig, "of_pairs": tot}).reset_index()
print("Bonferroni-significant group pairs per variant (out of 10):")
print(tab.to_string(index=False))
print("\nInterpretation: with ~9-17 subjects per ITA group, almost nothing survives")
print("Bonferroni. The heatmaps show DIRECTION; do not claim a proven per-group gap at n=28.")
FAIR_PAIR[FAIR_PAIR.significant][["model","group_a","group_b","bonferroni_p"]].round(4)

---
# E · ⭐ Bruise size

Small bruises are intrinsically harder — a few pixels of error costs proportionally
far more Dice on a small target. So a model's score partly reflects *which bruises*
were in the test set, and — crucially for section D — **if bruise size correlates with
skin tone, an apparent fairness gap can be a size effect in disguise.** This section
tests exactly that.

## E1 · Size distribution + Dice-vs-size + miss-vs-size

In [ ]:
sizes = per_image[CORE[0]].set_index("stem").gt_positive_pixels
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
axes[0].hist(np.log10(sizes.clip(lower=1)), bins=30, color="#2a78d6", alpha=0.85)
axes[0].set_xlabel("log10(bruise size, px)"); axes[0].set_ylabel("images")
axes[0].set_title(f"Bruise-size distribution (median {int(sizes.median()):,} px)", fontsize=10)
axes[0].grid(lw=0.5); axes[0].set_axisbelow(True)

for n in CORE:
    d = per_image[n]
    axes[1].scatter(d.gt_positive_pixels, d.dice, s=12, alpha=0.35, color=PALETTE[n], edgecolors="none")
axes[1].set_xscale("log"); axes[1].set_xlabel("bruise size (px, log)"); axes[1].set_ylabel("Dice")
axes[1].set_ylim(-0.02,1.02); axes[1].set_title("Dice vs size (all models)", fontsize=10)
axes[1].grid(lw=0.5); axes[1].set_axisbelow(True)

# miss rate by size quartile, per model
q = pd.qcut(sizes, 4, labels=["Q1 smallest","Q2","Q3","Q4 largest"])
size_bin = q.to_frame("size_q")
for n in CORE:
    d = per_image[n].set_index("stem")
    mr = d.join(size_bin).groupby("size_q", observed=True).complete_miss.mean()*100
    axes[2].plot(range(len(mr)), mr.values, "-o", color=PALETTE[n], label=DISP[n], lw=2)
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(["Q1\nsmallest","Q2","Q3","Q4\nlargest"], fontsize=8)
axes[2].set_ylabel("complete-miss rate (%)"); axes[2].set_title("Misses concentrate on small bruises", fontsize=10)
axes[2].grid(lw=0.5); axes[2].set_axisbelow(True); axes[2].legend(fontsize=6.5, frameon=False)
plt.tight_layout(); plt.show()
pd.DataFrame([{"model": DISP[n],
               "spearman_size_vs_dice": spearmanr(per_image[n].gt_positive_pixels, per_image[n].dice)[0],
               "p": spearmanr(per_image[n].gt_positive_pixels, per_image[n].dice)[1]} for n in CORE]).round(4)

## E2 · Recall & precision vs size

In [ ]:
q = pd.qcut(per_image[CORE[0]].set_index("stem").gt_positive_pixels, 5,
            labels=["Q1","Q2","Q3","Q4","Q5"])
qb = q.to_frame("size_q")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
for n in CORE:
    d = per_image[n].set_index("stem").join(qb)
    rec = d.groupby("size_q", observed=True).recall.mean()
    pre = d.groupby("size_q", observed=True).precision.mean()
    a1.plot(range(len(rec)), rec.values, "-o", color=PALETTE[n], lw=2, label=DISP[n])
    a2.plot(range(len(pre)), pre.values, "-o", color=PALETTE[n], lw=2)
for ax, t in [(a1,"Recall vs size quintile"),(a2,"Precision vs size quintile")]:
    ax.set_xticks(range(5)); ax.set_xticklabels(["Q1\nsmall","Q2","Q3","Q4","Q5\nlarge"], fontsize=8)
    ax.set_ylim(0,1.02); ax.set_title(t, fontsize=10.5); ax.grid(lw=0.5); ax.set_axisbelow(True)
a1.set_ylabel("mean recall"); a1.legend(fontsize=6.5, frameon=False, loc="lower right")
plt.tight_layout(); plt.show()
print("Recall falls fastest on the smallest bruises — that is where the misses live.")

## E3 · ⭐ The size↔fairness confound — bruise size by ITA group

In [ ]:
sz = per_image[CORE[0]][["stem","gt_positive_pixels"]].merge(ITA, on="stem")
groups = [g for g in GROUP_ORDER if g in sz.skin_tone_category.unique()]
fig, ax = plt.subplots(figsize=(9.5, 4.6))
data = [sz[sz.skin_tone_category==g].gt_positive_pixels.values for g in groups]
bp = ax.boxplot(data, labels=[g.split(" ")[0] for g in groups], showfliers=False, patch_artist=True)
for patch, c in zip(bp["boxes"], plt.cm.viridis(np.linspace(0,1,len(groups)))):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_yscale("log"); ax.set_ylabel("bruise size (px, log)")
ax.set_title("Bruise size by ITA group — is the fairness gap really a size gap?", fontsize=11, pad=10)
ax.grid(axis="y", lw=0.5); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

H, p = kruskal(*data)
tab = sz.groupby("skin_tone_category").gt_positive_pixels.agg(["count","median"]).reindex(groups)
print(f"Kruskal–Wallis (size across ITA groups): H={H:.2f}, p={p:.4f}")
print("If p is small, group-level Dice differences in section D are partly a size artefact,")
print("because smaller bruises are harder for every model (E1). Report size as a covariate.")
tab.round(0)

## E4 · ⭐ Does the skin-tone signal survive size? (stratified + regression)

E3 showed size is confounded with ITA group. This asks the decisive question: is the
light-skin deficit *only* size? Two tests per model — (1) within size terciles, compare
light vs rest on miss-rate and recall (if the gap is pure size, it vanishes once you
hold size roughly fixed); (2) a regression that puts `log10(size)` and a light-skin flag
in together — `recall ~ log_size + light` (OLS, always defined) and
`complete_miss ~ log_size + light` (logistic, where there are enough miss events). **SEs
are clustered by subject** (the light group is only ~9 people; treating 185 images as
independent would make these p-values far too small — the same reason every CI here is a
subject bootstrap). **If the `light` term stays significant with size controlled, the
signal is not only size; if it collapses, the E3 confound explains the section-D gap.**
Still n=28-limited — diagnostic, not proof.

In [ ]:
try:
    import statsmodels.formula.api as smf
    HAVE_SM = True
except Exception:
    HAVE_SM = False
    print("statsmodels unavailable -> regressions skipped (stratified table still runs)")

LIGHT = "Light (II-III)"
strat_rows, coef_rows = [], []
for n in CORE:
    d = per_image[n].merge(ITA, on="stem").merge(SUBJ, on="stem").copy()
    d["light"]    = (d.skin_tone_category == LIGHT).astype(int)
    d["log_size"] = np.log10(d.gt_positive_pixels.clip(lower=1))
    d["miss"]     = d.complete_miss.astype(int)
    d["size_tercile"] = pd.qcut(d.gt_positive_pixels, 3, labels=["small","medium","large"])

    # (1) stratified light-vs-rest within each size tercile (+ overall)
    for terc, sub in list(d.groupby("size_tercile", observed=True)) + [("all", d)]:
        L, R = sub[sub.light == 1], sub[sub.light == 0]
        strat_rows.append({"model": DISP[n], "size_tercile": str(terc),
                           "n_light": len(L), "n_rest": len(R),
                           "miss_light_pct": L.miss.mean()*100 if len(L) else np.nan,
                           "miss_rest_pct":  R.miss.mean()*100 if len(R) else np.nan,
                           "recall_light":   L.recall.mean() if len(L) else np.nan,
                           "recall_rest":    R.recall.mean() if len(R) else np.nan})

    # (2) regression: light term controlling for log size. SEs are CLUSTERED BY SUBJECT --
    # the 185 images are only 28 people, and the light group is ~9 subjects, so an
    # independence assumption would make these p-values far too small (the same reason
    # every CI in this notebook is a subject bootstrap). Clustered SEs are the honest test.
    if HAVE_SM:
        clk = {"cov_type": "cluster", "cov_kwds": {"groups": d["subject"]}}
        try:
            ols = smf.ols("recall ~ log_size + light", data=d).fit(**clk)
            coef_rows.append({"model": DISP[n], "outcome": "recall (OLS, subj-clustered)",
                              "light_coef": ols.params["light"], "light_p": ols.pvalues["light"],
                              "logsize_coef": ols.params["log_size"], "logsize_p": ols.pvalues["log_size"],
                              "light_OR": np.nan})
        except Exception:
            coef_rows.append({"model": DISP[n], "outcome": "recall (OLS): failed", "light_coef": np.nan,
                              "light_p": np.nan, "logsize_coef": np.nan, "logsize_p": np.nan, "light_OR": np.nan})
        # Logistic needs enough miss EVENTS or it quasi-separates and returns garbage
        # (e.g. a model with 1 miss gives OR ~1e10). recall(OLS) carries the signal for
        # the near-zero-miss SegFormer models; the miss logit is only run where it's stable.
        if int(d.miss.sum()) >= 5:
            try:
                import warnings
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    lg = smf.logit("miss ~ log_size + light", data=d).fit(disp=0, **clk)
                coef_rows.append({"model": DISP[n], "outcome": "miss (Logit)",
                                  "light_coef": lg.params["light"], "light_p": lg.pvalues["light"],
                                  "logsize_coef": lg.params["log_size"], "logsize_p": lg.pvalues["log_size"],
                                  "light_OR": float(np.exp(lg.params["light"]))})
            except Exception:
                coef_rows.append({"model": DISP[n], "outcome": "miss (Logit: failed)",
                                  "light_coef": np.nan, "light_p": np.nan, "logsize_coef": np.nan,
                                  "logsize_p": np.nan, "light_OR": np.nan})

SIZE_COND      = pd.DataFrame(strat_rows)
SIZE_COND_COEF = pd.DataFrame(coef_rows)

# figure: miss% by size tercile, light vs rest, for YOLO-direct (where the gap lives)
focus = "yolo_sem_direct"
fd = SIZE_COND[(SIZE_COND.model == DISP[focus]) & (SIZE_COND.size_tercile != "all")]
fig, ax = plt.subplots(figsize=(8, 4.4))
x = np.arange(len(fd)); w = 0.38
ax.bar(x - w/2, fd["miss_light_pct"], w, color="#e34948", label="Light (II-III)")
ax.bar(x + w/2, fd["miss_rest_pct"],  w, color="#2a78d6", label="rest")
for xi, vl, vr in zip(x, fd["miss_light_pct"], fd["miss_rest_pct"]):
    ax.text(xi - w/2, (vl if np.isfinite(vl) else 0)+0.3, f"{vl:.0f}", ha="center", fontsize=8)
    ax.text(xi + w/2, (vr if np.isfinite(vr) else 0)+0.3, f"{vr:.0f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(fd.size_tercile)
ax.set_ylabel("complete-miss rate (%)"); ax.set_xlabel("bruise-size tercile")
ax.set_title(f"{DISP[focus]}: is the light-skin miss gap just size?", fontsize=11, pad=10)
ax.legend(frameon=False); ax.grid(axis="y", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

print("Stratified light-vs-rest by size tercile (miss% and recall):")
print(SIZE_COND.round(2).to_string(index=False))
print("\nRegression — 'light' term with log10(size) controlled:")
print(SIZE_COND_COEF.round(4).to_string(index=False))
print("\nRead: light_p<0.05 with log_size in the model => skin-tone signal is NOT only size.")
print("      light term collapsing to n.s. => the E3 size confound explains the section-D gap.")
SIZE_COND_COEF.round(4)

---
# F · Annotation ceiling & cost

## F1 · ⭐ The annotation ceiling — do models beat human–human agreement?

Our test masks are a **majority vote of three experts**. How much do the experts even
agree with each other? We read the per-image inter-labeler Dice from
`interlabeler_agreement_640.csv` (pure mask arithmetic, no model). If experts agree at
only ~0.64, a model at ~0.79 vs their consensus is already **past the point where
humans agree with each other** — and a 0.02 Dice gap between two models is far below
the label noise. Note **Paul** (whose masks we *train on*) is the outlier: he matches
the majority least of the three.

In [ ]:
IL = pd.read_csv(WORK / "interlabeler_agreement_640.csv")
human_pairs = ["paul_vs_gbarimah","paul_vs_erik","gbarimah_vs_erik"]
human_maj   = ["paul_vs_majority","gbarimah_vs_majority","erik_vs_majority"]
rows = []
for c in human_pairs:
    a, b = c.split("_vs_"); rows.append({"label": f"{a.capitalize()} vs {b.capitalize()}", "kind":"human vs human", "dice": IL[c].mean()})
for c in human_maj:
    a = c.split("_vs_")[0]; rows.append({"label": f"{a.capitalize()} vs majority", "kind":"human vs majority", "dice": IL[c].mean()})
for n in CORE:
    rows.append({"label": DISP[n], "kind":"model vs majority", "dice": per_image[n].dice.mean()})
ceiling = pd.DataFrame(rows)
HH = IL[human_pairs].values.mean()

KIND_C = {"human vs human":"#e34948", "human vs majority":"#eb6834", "model vs majority":"#2a78d6"}
fig, ax = plt.subplots(figsize=(9, 5.4))
y = np.arange(len(ceiling))[::-1]
ax.barh(y, ceiling.dice, height=0.62, color=[KIND_C[k] for k in ceiling.kind], zorder=3)
for yy, v in zip(y, ceiling.dice):
    ax.text(v+0.008, yy, f"{v:.3f}", va="center", fontsize=8.5)
ax.axvline(HH, color=MUTED, ls="--", lw=1.5, zorder=2)
ax.text(HH, len(ceiling)-0.2, f"  avg human-human = {HH:.3f}", color=MUTED, fontsize=9, va="top")
ax.set_yticks(y); ax.set_yticklabels(ceiling.label, fontsize=8.5)
ax.set_xlabel("Mean Dice"); ax.set_xlim(0, 1.05)
ax.set_title("Annotation ceiling: every model beats human–human agreement", fontsize=12, pad=10)
ax.grid(axis="x", lw=0.6, zorder=0); ax.set_axisbelow(True)
ax.legend(handles=[Line2D([0],[0],marker="s",lw=0,markerfacecolor=c,markeredgecolor="none",markersize=9,label=k)
                   for k,c in KIND_C.items()], loc="upper center", bbox_to_anchor=(0.5,-0.09), ncol=3, fontsize=8.5, frameon=False)
plt.tight_layout(); plt.show()
ceiling.round(4)

## F2 · Does the model beat the annotator it learned from?

Models train on **Paul's** masks only, scored vs the 3-expert majority. Paul himself
matches that majority at ~0.70. Paired subject-bootstrap Δ (model − Paul, vs the same
majority): if the interval clears zero, the model trained on Paul agrees with consensus
**better than Paul does** — training averages away one annotator's idiosyncrasies.

In [ ]:
base = IL[["stem","paul_vs_majority"]].merge(SUBJ, on="stem")
rows = []
for n in CORE:
    m = base.merge(per_image[n][["stem","dice"]].rename(columns={"dice":"model"}), on="stem")
    r = paired_delta(m, lambda x: x.model.mean() - x.paul_vs_majority.mean(), B=4000)
    rows.append({"model": n, "model_vs_maj": m.model.mean(), "paul_vs_maj": m.paul_vs_majority.mean(),
                 "delta": r["delta"], "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"], "p_gt0": r["p_gt0"],
                 "beats_paul": r["ci_lo"] > 0})
beat = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8.8, 4.2))
xs = np.arange(len(beat))
ax.bar(xs, beat.delta, width=0.55, color=[PALETTE[n] for n in beat.model], zorder=3)
ax.errorbar(xs, beat.delta, yerr=[beat.delta-beat.ci_lo, beat.ci_hi-beat.delta],
            fmt="none", ecolor=INK, elinewidth=1.6, capsize=5, zorder=4)
ax.axhline(0, color="#c3c2b7", lw=1.2, zorder=2)
for x, d_, hi, p in zip(xs, beat.delta, beat.ci_hi, beat.p_gt0):
    ax.text(x, hi+0.005, f"{d_:+.3f}\nP={p*100:.0f}%", ha="center", fontsize=7.5)
ax.set_xticks(xs); ax.set_xticklabels([DISP[n] for n in beat.model], rotation=18, ha="right", fontsize=7.5)
ax.set_ylabel("Δ mean Dice (model − Paul, vs majority)")
ax.set_title("Every model matches expert consensus better than the annotator it trained on", fontsize=10.5, pad=10)
ax.grid(axis="y", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
beat.assign(model=lambda d: d.model.map(DISP)).round(4)

## F3 · Speed / cost — accuracy vs latency

From `results_final/benchmark_640.csv` (640-tensor-on-GPU → mask-on-GPU, seed 0, the
architectural number). Pareto view: up-and-left is better. YOLO is far faster but pays
in miss rate; SegFormer-B0 is the balance point.

In [ ]:
try:
    B = pd.read_csv(OUT_DIR/"benchmark_640.csv")
    name_map = {"segformer_b2_teacher":"segformer_b2_teacher","segformer_b0_direct":"segformer_b0_direct",
                "segformer_b0_distilled":"segformer_b0_distilled","yolo_sem_direct":"yolo_sem_direct",
                "yolo_sem_distilled":"yolo_sem_distilled"}
    fig, ax = plt.subplots(figsize=(8.6, 5))
    for _, r in B.iterrows():
        n = r["model"]
        if n not in per_image: continue
        med = per_image[n].dice.median()
        ax.scatter(r.fps, med, s=180, color=PALETTE[n], edgecolors=INK, lw=0.7, zorder=3)
        ax.annotate(f"{DISP[n]}\n{r.params_M:.1f}M · p95 {r.p95_ms:.1f}ms",
                    (r.fps, med), fontsize=7.5, textcoords="offset points", xytext=(8,-4))
    ax.set_xlabel("throughput (FPS, higher better)"); ax.set_ylabel("median Dice (higher better)")
    ax.set_title("Accuracy vs speed — best seed Dice × benchmarked FPS", fontsize=11, pad=10)
    ax.grid(lw=0.6); ax.set_axisbelow(True)
    plt.tight_layout(); plt.show()
    print(B[["model","median_ms","p95_ms","fps","params_M"]].to_string(index=False))
except Exception as e:
    print("benchmark_640.csv not read:", e)

---
# G · Qualitative (optional GPU)

Reloads native images + re-predicts a handful of masks to render overlays. Uses the
best-seed model handles already in memory, so it's cheap. Set `CFG["render_gallery"]
= False` in §1 to skip. Self-skips if the images aren't present.

In [ ]:
if not CFG.get("render_gallery", True):
    print("gallery skipped (CFG['render_gallery'] = False)")
else:
    IMG_H = IMG_W = CFG["img_size"]
    def load_rgb640(p):
        im = cv2.imread(str(p));
        return cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2RGB), (IMG_W, IMG_H)) if im is not None else None
    def to640(mask):
        m = np.asarray(mask)
        if m.ndim == 3: m = m[...,0]
        return (cv2.resize(m.astype("uint8"), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST) > 0).astype("uint8")
    def overlay(img, mask, color=(230,60,60), alpha=0.45):
        lay = np.zeros_like(img); lay[mask.astype(bool)] = color
        return cv2.addWeighted(lay, alpha, img, 1-alpha, 0)

    def predict_mask(name, img_path):
        if name in SEG_MODELS:
            model, cut = SEG_MODELS[name]
            rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            r = cv2.resize(rgb, (IMG_W, IMG_H)).astype(np.float32)/255.0
            t = torch.from_numpy(r.transpose(2,0,1)).unsqueeze(0).float().to(DEVICE)
            with torch.no_grad():
                return (model(t)[:,0] >= cut).to(torch.uint8)[0].cpu().numpy()
        from ultralytics import YOLO
        res = YOLO(str(YOLO_BEST[name])).predict(str(img_path), imgsz=IMG_H, device="0", verbose=False)[0]
        cm = res.semantic_mask.data if getattr(res,"semantic_mask",None) is not None else np.zeros((IMG_H,IMG_W))
        cm = cm.cpu().numpy() if hasattr(cm,"cpu") else np.asarray(cm)
        return to640((cm==1).astype("uint8"))

    D = pd.DataFrame({n: per_image[n].set_index("stem").dice for n in CORE})
    mean_d = D.mean(axis=1).sort_values()
    picks = [mean_d.index[i] for i in [len(mean_d)-1, int(len(mean_d)*0.6), int(len(mean_d)*0.25), 0]]
    labels = ["easiest","typical","hard","hardest"]
    tdf = MAN["test"].set_index("stem")
    fig, axes = plt.subplots(len(picks), len(CORE)+1, figsize=(3*(len(CORE)+1), 3*len(picks)))
    for i, (stem, lab) in enumerate(zip(picks, labels)):
        row = tdf.loc[stem]
        img = load_rgb640(WORK/row.image_path)
        gt  = to640(cv2.imread(str(WORK/row.mask_path), cv2.IMREAD_GRAYSCALE))
        axes[i,0].imshow(overlay(img, gt, (40,190,40))); axes[i,0].set_ylabel(f"{lab}\n{stem}", fontsize=7)
        if i==0: axes[i,0].set_title("Ground truth", fontsize=9)
        for j, n in enumerate(CORE, start=1):
            m = predict_mask(n, WORK/row.image_path)
            axes[i,j].imshow(overlay(img, m))
            axes[i,j].set_xlabel(f"Dice {per_image[n].set_index('stem').dice[stem]:.2f}", fontsize=7.5)
            if i==0: axes[i,j].set_title(DISP[n], fontsize=8)
    for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
    fig.suptitle("Predictions across easy → hard (green = truth, red = model)", fontsize=12, y=1.01)
    plt.tight_layout(); plt.show()

In [ ]:
if CFG.get("render_gallery", True):
    # Failure gallery: images where YOLO-direct returns EMPTY but SegFormer-B0-distilled finds it.
    yd = per_image["yolo_sem_direct"]
    misses = yd[yd.complete_miss].stem.tolist()[:3]
    tdf = MAN["test"].set_index("stem")
    if not misses:
        print("No complete misses for YOLO-direct in this best seed.")
    else:
        fig, axes = plt.subplots(len(misses), 3, figsize=(9, 3*len(misses))); axes = np.atleast_2d(axes)
        for i, stem in enumerate(misses):
            row = tdf.loc[stem]
            img = load_rgb640(WORK/row.image_path)
            gt  = to640(cv2.imread(str(WORK/row.mask_path), cv2.IMREAD_GRAYSCALE))
            y_m = predict_mask("yolo_sem_direct", WORK/row.image_path)
            s_m = predict_mask("segformer_b0_distilled", WORK/row.image_path)
            axes[i,0].imshow(overlay(img, gt, (40,190,40))); axes[i,0].set_ylabel(stem, fontsize=7)
            axes[i,1].imshow(overlay(img, y_m)); axes[i,2].imshow(overlay(img, s_m))
            axes[i,1].set_xlabel(f"{int(y_m.sum())} px", fontsize=7.5)
            axes[i,2].set_xlabel(f"Dice {per_image['segformer_b0_distilled'].set_index('stem').dice[stem]:.2f}", fontsize=7.5)
            if i==0:
                axes[i,0].set_title("Ground truth", fontsize=9)
                axes[i,1].set_title("YOLO-direct — EMPTY", fontsize=9, color="#d03b3b")
                axes[i,2].set_title("SegFormer-B0 distilled", fontsize=9)
        for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
        fig.suptitle("Complete misses: a real bruise, model returns nothing", fontsize=12, y=1.01)
        plt.tight_layout(); plt.show()

## G2 · ⭐ Overlays by skin-tone (ITA) group

One representative image **per ITA group** — the image whose SegFormer-B0-distilled Dice
is closest to that group's median (the "typical" case, not a cherry-picked best/worst).
Columns: ground truth, the safe model (B0-distilled), and the miss-prone one
(YOLO-direct). This is the visual companion to sections D/E: eyeball how the two models
behave as skin tone darkens (and remember from E3 that light-skin bruises are also the
smallest, so tone and size move together here).

In [ ]:
if CFG.get("render_gallery", True):
    ITA_ORDER = ["Light (II-III)","Intermediate (III-IV)","Tan (IV)","Brown (V)","Dark (VI)"]
    groups = [g for g in ITA_ORDER if g in ITA.skin_tone_category.unique()]
    ref = "segformer_b0_distilled"
    dref = per_image[ref].set_index("stem").dice
    tdf  = MAN["test"].merge(ITA, on="stem").set_index("stem")
    COLS = [("Ground truth", None),
            (DISP["segformer_b0_distilled"], "segformer_b0_distilled"),
            (DISP["yolo_sem_direct"], "yolo_sem_direct")]
    picks = []
    for g in groups:
        stems = ITA[ITA.skin_tone_category == g].stem
        dd = dref.loc[dref.index.isin(stems)]
        picks.append(dd.sub(dd.median()).abs().idxmin())   # image closest to the group's median Dice
    fig, axes = plt.subplots(len(picks), 3, figsize=(9.5, 3*len(picks))); axes = np.atleast_2d(axes)
    for i, (g, stem) in enumerate(zip(groups, picks)):
        row = tdf.loc[stem]; img = load_rgb640(WORK/row.image_path)
        gt = to640(cv2.imread(str(WORK/row.mask_path), cv2.IMREAD_GRAYSCALE))
        for j, (title, mdl) in enumerate(COLS):
            if mdl is None:
                axes[i,j].imshow(overlay(img, gt, (40,190,40)))
            else:
                m = predict_mask(mdl, WORK/row.image_path)
                axes[i,j].imshow(overlay(img, m))
                axes[i,j].set_xlabel(f"Dice {per_image[mdl].set_index('stem').dice[stem]:.2f}", fontsize=7.5)
            if i == 0: axes[i,j].set_title(title, fontsize=8.5)
        axes[i,0].set_ylabel(f"{g.split(' ')[0]}\n{int(row.gt_positive_pixels) if 'gt_positive_pixels' in row else ''}", fontsize=7)
    for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
    fig.suptitle("Typical prediction per ITA skin-tone group (green = truth, red = model)", fontsize=11.5, y=1.01)
    plt.tight_layout(); plt.show()

## G3 · ⭐ Overlays by bruise-size category

Same idea, binned by **bruise size** instead of skin tone: one typical image per size
quartile (Q1 smallest → Q4 largest), same three columns. This makes the E1/E2 finding
concrete — watch the small-bruise row, where YOLO is most likely to blank while
SegFormer still finds something.

In [ ]:
if CFG.get("render_gallery", True):
    ref = "segformer_b0_distilled"
    sz = per_image[ref][["stem","gt_positive_pixels"]].copy()
    sz["bin"] = pd.qcut(sz.gt_positive_pixels, 4, labels=["Q1 smallest","Q2","Q3","Q4 largest"])
    tdf = MAN["test"].set_index("stem")
    COLS = [("Ground truth", None),
            (DISP["segformer_b0_distilled"], "segformer_b0_distilled"),
            (DISP["yolo_sem_direct"], "yolo_sem_direct")]
    bins = ["Q1 smallest","Q2","Q3","Q4 largest"]
    picks = []
    for b in bins:
        s = sz[sz.bin == b].set_index("stem").gt_positive_pixels
        picks.append(s.sub(s.median()).abs().idxmin())     # image with the bin's median size
    fig, axes = plt.subplots(len(picks), 3, figsize=(9.5, 3*len(picks))); axes = np.atleast_2d(axes)
    for i, (b, stem) in enumerate(zip(bins, picks)):
        row = tdf.loc[stem]; img = load_rgb640(WORK/row.image_path)
        gt = to640(cv2.imread(str(WORK/row.mask_path), cv2.IMREAD_GRAYSCALE))
        px = int(sz.set_index("stem").gt_positive_pixels[stem])
        for j, (title, mdl) in enumerate(COLS):
            if mdl is None:
                axes[i,j].imshow(overlay(img, gt, (40,190,40)))
            else:
                m = predict_mask(mdl, WORK/row.image_path)
                axes[i,j].imshow(overlay(img, m))
                axes[i,j].set_xlabel(f"Dice {per_image[mdl].set_index('stem').dice[stem]:.2f}", fontsize=7.5)
            if i == 0: axes[i,j].set_title(title, fontsize=8.5)
        axes[i,0].set_ylabel(f"{b}\n{px:,} px", fontsize=7)
    for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
    fig.suptitle("Typical prediction per bruise-size quartile (green = truth, red = model)", fontsize=11.5, y=1.01)
    plt.tight_layout(); plt.show()

## Save every table + figure to Drive

In [ ]:
import datetime, shutil
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
ADIR = Path(CFG["drive_dir"]) / f"final_analysis_{stamp}"
ADIR.mkdir(parents=True, exist_ok=True)
for n in CORE:
    per_image[n].merge(SUBJ, on="stem").merge(ITA, on="stem").to_csv(ADIR/f"per_image_{n}.csv", index=False)
for n in YOLO_MODELS:
    per_image_custom[n].to_csv(ADIR/f"per_image_{n}_custom255.csv", index=False)
head.to_csv(ADIR/"headline.csv", index=False)
FOREST.to_csv(ADIR/"paired_contrasts.csv", index=False)
miss.to_csv(ADIR/"miss_rates.csv", index=False)
beat.to_csv(ADIR/"model_vs_paul.csv", index=False)
ceiling.to_csv(ADIR/"annotation_ceiling.csv", index=False)
FAIR_GROUP.to_csv(ADIR/"fairness_per_group_bestseed.csv", index=False)
FAIR_STATS.to_csv(ADIR/"fairness_stats_bestseed.csv", index=False)
FAIR_PAIR.to_csv(ADIR/"fairness_pairwise_bestseed.csv", index=False)
SIZE_COND.to_csv(ADIR/"size_conditioned_fairness_stratified.csv", index=False)
SIZE_COND_COEF.to_csv(ADIR/"size_conditioned_fairness_regression.csv", index=False)
print("saved ->", ADIR)
for f in sorted(ADIR.glob("*")): print("   ", f.name)

### How to read this analysis

1. **Best-seed, one inference pass** — every figure shares it; 3-seed spread is on disk.
2. **Miss rate is the honest axis** (B1): it separates models by more than label noise.
3. **Fairness (D) is exploratory** at n=28 — direction, not proof; and **size (E3) is a
   confound** you must report as a covariate before claiming a skin-tone gap.
4. **The ceiling (F1) reframes everything**: model-to-model Dice gaps live below the
   ~0.36 of annotation noise, and a model trained on Paul beats Paul vs consensus (F2).